# AI Hub 조합 데이터 EDA (외부 데이터 보충)

## EDA의 목적
- 알약 검출 모델의 학습 데이터를 AI Hub 576 "경구약제 이미지" 허용 조합 데이터로 보충하기에 앞서, 외부 데이터의 형태, 라벨 품질, 분포를 대회 원본 EDA와 같은 순서로 확인한다.
- 본 EDA를 통해 AI Hub 라벨의 변환 방법과 정제 목록을 확정하고, 어떤 AI Hub 이미지를 학습에 넣을지 정한다.
- 각 섹션의 분석 결과를 보충 데이터 변환, 라벨 정제, 입력 크기와 증강, 클래스 체계, 보충 이미지 선택, 평가 방법에 대한 결정으로 연결하고, 7번 EDA 결론에서 결정 사항 표로 정리한다.
- 대회 원본 분석과 원본 라벨 정제는 [00_raewoo_first_eda.ipynb](00_raewoo_first_eda.ipynb)에 있다.
    - 이 노트북은 00이 저장한 원본 정제 목록(`00_raewoo_label_fixes.csv`)과 test 기록표(`00_raewoo_test_review.csv`)를 읽어 쓴다. 00을 먼저 실행하지 않아도 된다.
- 학습용 분할은 이 노트북에서 정하지 않는다. (팀원 작업)
- 데이터 준비 (README의 AI Hub 외부 데이터 받기)
    - 대회에서 금지한 `TL_2_조합.zip`, `TS_2_조합.zip`은 받지 않는다. (RULE.md)
    - 라벨 zip 8개(약 59MB)를 받아 푼다.
        - `uv run python script/download_aihub.py 66065 66067 66068 66069 66070 66071 66072 66243`
        - `uv run python script/extract_aihub.py data/external/aihub/train_annotations/*.zip`
    - 이미지 zip 8개(각 약 3GB)는 디스크를 아끼려고 하나씩 받고, 풀고, 지운다.
        - `uv run python script/download_aihub.py 66154`
        - `uv run python script/extract_aihub.py data/external/aihub/train_images/TS_1_조합.zip --delete-images`
        - filekey 66156~66161(TS_3~8), 66254(VS_1)도 같은 방법으로 받는다.

## EDA 목차
> 각 분석 결과 아래에는 **인사이트 / 결론** 마크다운 셀을 두어, 분석 결과와 그에 따른 결정을 기록한다.

0. 사전 설정
    - seed, 데이터 경로, 그림 저장 경로, 한글 폰트, 도우미 함수를 정의한다.
    - 대회 원본 라벨, 00의 정제 목록과 test 기록표, AI Hub 라벨 zip을 읽고, 모든 이미지의 크기·밝기·축소 그림을 한 번에 계산해 둔다.
1. 데이터셋 훑어보기
    - **파일 구성과 금지 범위**
        - AI Hub 576 데이터셋에서 쓸 수 있는 파일과 금지된 파일을 정리하고, 받은 라벨과 이미지의 수, 해상도, 깨진 파일을 확인한다.
    - **라벨 구조 비교**
        - AI Hub 원본 JSON과 Kaggle JSON의 폴더 구조, 키 구성, id 값, 클래스 표기를 비교하고, 같은 알약의 JSON 두 개를 키마다 나란히 놓는다.
    - **Kaggle 형식으로 바꾼 뒤 다시 비교**
        - AI Hub 라벨을 Kaggle 형식으로 바꾸고, 같은 비교를 다시 해 사진에 따라 다른 값만 남는지 확인한다.
            - 목적: AI Hub 라벨을 Kaggle 라벨과 같은 코드로 읽을 수 있게 바꾸는 방법을 정한다. 이후 분석은 바꾼 라벨로 한다.
    - **촬영 조건 비교**
        - 메타데이터의 촬영 조건과, AI Hub·대회 train·test의 채널별 평균 밝기를 비교한다.
            - 목적: AI Hub 사진을 대회 사진과 그대로 섞어 학습해도 되는지 정한다.
    - **이미지 육안 확인**
        - zip마다 1장씩 8장에 box를 겹쳐 그린다. (seed 고정)
            - 목적: 바꾼 box가 알약을 제대로 감싸는지 확인한다.
    - **누수 점검**
        - 파일 이름, 파일 내용, 축소 그림으로 AI Hub와 대회 train·test, AI Hub zip끼리 같은 사진이 있는지 찾는다.
            - 목적: 대회 데이터와 겹치는 사진을 학습에서 뺄지 정하고, 대회 데이터의 출처를 확인한다.
2. 라벨 품질
    - **라벨 누락과 클래스 불일치**
        - 파일명의 K-코드와 bbox의 category_id를 대조하고, 빠진 라벨을 원인별로 나눠 규모와 예시를 본다.
        - 같은 K-코드의 JSON끼리 알약 정보(제품명, 각인)가 같은지 확인한다.
    - **좌표 오류와 고치는 방법**
        - 이미지 밖 bbox, area 오타, 다른 클래스와 같은 bbox, 숫자가 붙은 bbox를 찾는다.
        - 빠진 box를 OpenCV로 사진에서 찾아 그려 넣는 방법을 만들고, 라벨이 있는 box로 정확도를 먼저 확인한다.
        - `-오류` 폴더의 JSON이 왜 오류인지 사진으로 확인한다.
    - **클래스별 crop 갤러리**
        - 모든 bbox를 클래스별로 잘라 시트 여러 장에 나눠 저장하고, 잘못된 클래스와 헐거운 box를 찾는다.
            - 대회 train과 같은 클래스는 대회 crop을 첫 줄에 놓아 같은 알약인지 대조한다.
    - **라벨 정제 목록**
        - 2-1~2-3에서 찾은 라벨 오류를 모아 처리 방법을 정하고 CSV(`01_raewoo_aihub_label_fixes.csv`)로 저장한다.
    - **라벨 정제 처리**
        - 정제 목록을 적용한 AI Hub 라벨을 만든다. 3번부터는 정제 라벨로 본다.
            - 목적: 고치거나 뺄 라벨과 이미지를 정한다.
            - 참고: 평가 지표 mAP@[0.75:0.95]는 IoU 0.75 미만인 box를 오답으로 처리하므로, box 오차에 매우 민감하다. (00 8-2)
3. bbox 분포
    - **객체 수와 크기**
        - 이미지당 객체 수, box 크기를 대회 train과 비교하고, 같은 알약의 box 크기(여백)가 대회 라벨과 같은지 본다.
            - 목적: 00에서 정한 입력 크기와 증강이 AI Hub를 섞은 뒤에도 맞는지 확인하고, 라벨 여백 차이로 점수가 깎이지 않는지 확인한다.
    - **위치와 겹침**
        - box 중심 위치와, 같은 이미지 안 box끼리의 IoU를 대회 train과 비교한다.
            - 목적: 00에서 정한 후처리 설정(NMS IoU, 최대 검출 수)이 그대로 맞는지 확인한다.
4. 클래스 분포
    - **클래스별 객체 수와 등장 조합 수**
        - AI Hub의 클래스별 객체 수와 조합 수를 대회 train 클래스와 train에 없는 클래스로 나눠 본다.
    - **조합당 이미지 수**
        - 조합 하나에 사진이 몇 장인지와 촬영 각도 분포를 확인한다.
    - **같은 조합 사진 비교**
        - 같은 조합의 각도별 사진 3장을 나란히 본다.
    - **대회 train과 합친 분포**
        - 정제한 대회 train과 AI Hub를 합쳤을 때의 클래스 분포와 불균형 정도를 본다.
    - **가장 균일하게 고르기**
        - 클래스별 목표 객체 수를 채우는 이미지를 고르고, 전부 쓸 때와 클래스 분포를 비교한다.
            - 목적: 학습 클래스 체계와 학습에 넣을 AI Hub 이미지, 클래스 불균형 대응 방법을 정한다.
5. test 전용 클래스 채우기
    - **test에서 읽은 각인 대조**
        - 00 5-3 기록표의 각인을 AI Hub 라벨의 각인 필드에서 찾는다.
    - **test 알약과 AI Hub 알약 나란히 보기**
        - train에 없는 test 알약의 사진과, 각인으로 찾은 AI Hub 알약의 crop을 나란히 놓고 같은 알약인지 눈으로 대조한다.
    - **test 사진 단위 커버리지**
        - 모든 알약이 학습 클래스에 있는 test 사진의 비율을 보충 전후로 센다.
            - 목적: test에만 있는 알약이 보충으로 얼마나 채워지는지 확인하고, 남는 알약을 정한다.
6. 평가 기준선
    - **변환 라벨로 mAP 계산 검증**
        - 바꾼 AI Hub 라벨을 정답과 예측으로 함께 넣었을 때 mAP@[0.75:0.95]가 1.0인지 확인한다.
            - 목적: 118개 클래스 체계로도 로컬 평가 코드가 맞게 동작하는지 확인한다.
7. EDA 결론
    - 1~6번의 결과를 보충 데이터 변환, 라벨 정제, 클래스 체계, 보충 이미지 선택, 클래스 불균형, 평가 방법 항목별 결정 사항 표로 정리한다.

## 0. 사전 설정

In [ ]:
import collections
import concurrent.futures
import contextlib
import csv
import hashlib
import io
import json
import os
import shutil
import sys
import unicodedata
import zipfile
from pathlib import Path

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize
from IPython.display import Markdown, display
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# AI Hub 라벨 읽기와 변환은 공용 모듈의 함수를 쓴다 (script/extract_aihub.py와 같은 규칙)
from pill_detection.data import combo_name, convert_aihub_to_kaggle, load_annotations, pill_codes, read_aihub_label_zip

# 실행 환경의 버전 확인 (팀원 간 환경 차이로 오류가 생겼을 때 원인 파악용)
print(f"Python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")
print(f"OpenCV       : {cv2.__version__}")

In [ ]:
# 랜덤 샘플 시각화 등의 재현성을 위한 seed 고정
SEED = 42
np.random.seed(SEED)

# 노트북을 어디서 실행해도 레포 최상위(pyproject.toml이 있는 폴더)를 기준으로 경로를 잡는다
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TRAIN_IMG_DIR = RAW_DIR / 'train_images'
TRAIN_ANN_DIR = RAW_DIR / 'train_annotations'
TEST_IMG_DIR = RAW_DIR / 'test_images'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
AIHUB_DIR = EXTERNAL_DIR / 'aihub'  # script/download_aihub.py로 받은 zip
AIHUB_RAW_ANN_DIR = EXTERNAL_DIR / 'aihub_raw' / 'train_annotations'  # 라벨 zip을 그대로 푼 원본 JSON
AIHUB_ANN_DIR = EXTERNAL_DIR / 'aihub_converted' / 'train_annotations'  # Kaggle 형식으로 바꾼 JSON (1-3에서 만든다)
AIHUB_IMG_DIR = EXTERNAL_DIR / 'aihub_converted' / 'train_images'  # 이미지 zip의 PNG
AIHUB_SOURCES_CSV = EXTERNAL_DIR / 'aihub_converted' / 'train_images_sources.csv'  # 이미지마다 나온 zip
EDA_DIR = PROJECT_ROOT / 'notebooks' / 'eda'
# 그림과 계산 결과 저장 경로 (runs/는 git에서 제외됨)
FIG_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'figures_01'
CACHE_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'cache'
FIG_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 대회와 AI Hub 이미지 크기 (너비, 높이)
IMG_W, IMG_H = 976, 1280

# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트 사용 (macOS, Windows, Linux 순)
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ['AppleGothic', 'Malgun Gothic', 'NanumGothic']:
    if font_name in installed_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트에서 마이너스 기호가 깨지는 문제 방지


def save_fig(fig, name):
    """그림을 FIG_DIR에 png 파일로 저장한다."""
    fig.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')


def show_table(header, rows):
    """열 이름 리스트와 행 리스트를 마크다운 표로 출력한다."""
    lines = ['| ' + ' | '.join(header) + ' |', '|' + ' --- |' * len(header)]
    # 값 속 |(예: 분할선 각인 K|50)가 칸 구분자로 읽히지 않게 \|로 바꾼다
    lines += ['| ' + ' | '.join(str(value).replace('|', r'\|') for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


def box_iou(box_a, box_b):
    """두 xywh box의 IoU(겹친 넓이 ÷ 합친 넓이, 1이면 완전히 같음)를 계산한다."""
    inter_w = max(0, min(box_a[0] + box_a[2], box_b[0] + box_b[2]) - max(box_a[0], box_b[0]))
    inter_h = max(0, min(box_a[1] + box_a[3], box_b[1] + box_b[3]) - max(box_a[1], box_b[1]))
    inter = inter_w * inter_h
    return inter / (box_a[2] * box_a[3] + box_b[2] * box_b[3] - inter)


def crop_square(image, box, pad=0.1, size=224):
    """box 중심에서 긴 변보다 pad만큼 큰 정사각형을 잘라 size×size로 줄인다. 알약 모양이 늘어나지 않는다."""
    x, y, w, h = box
    side = max(w, h) * (1 + pad)
    cx, cy = x + w / 2, y + h / 2
    return image.crop((int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))).resize((size, size))


def pill_label(ann):
    """box 위에 적을 K-코드와 각인(메타데이터 print_front/print_back)을 만든다."""
    marks = [ann['image'][key] for key in ('print_front', 'print_back') if ann['image'].get(key)]
    return f"K-{ann['category_id']:06d} ({'/'.join(marks) or '각인 없음'})"


print(f"레포 최상위  : {PROJECT_ROOT}")
print(f"그래프 폰트  : {plt.rcParams['font.family'][0]}")
print(f"그림 저장    : {FIG_DIR}")

- 대회 원본 라벨과 00의 결과 파일을 읽는다.
    - `clean_records`: 00 2-5의 정제 목록(`00_raewoo_label_fixes.csv`)을 원본 라벨에 적용한 대회 train 라벨이다. (이미지 232장, 객체 771개)
        - 그려 넣은 box(`bbox_before`가 빈 줄)의 알약 메타데이터는 같은 클래스의 다른 annotation에서 가져온다. 00 2-5와 같은 방법이다.
    - `review_rows`: 00 5-2에서 test 사진을 보고 적은 기록표이다. train에 없는 알약 한 개가 한 줄이다.
- AI Hub 라벨 zip 8개를 `read_aihub_label_zip`(공용 모듈)으로 이미지 단위로 읽는다.
    - 이 함수는 category_id를 drug_N(K-코드)에서 만들고, 학습에 바로 쓰면 안 되는 이미지에 이유(`errors`)를 적는다. (1-2, 2-1)

In [ ]:
train_records = load_annotations(TRAIN_ANN_DIR)
train_files = sorted(os.listdir(TRAIN_IMG_DIR))
test_files = sorted(os.listdir(TEST_IMG_DIR), key=lambda name: int(name.split('.')[0]))
train_classes = sorted({a['category_id'] for r in train_records.values() for a in r['annotations']})
train_combos = sorted({combo_name(fn) for fn in train_files})
assert set(train_records) == set(train_files), '라벨이 있는 이미지와 이미지 파일 목록이 다르다'

# 00 2-5의 정제 목록을 원본 라벨에 적용한다
IMAGE_FIELDS = ['back_color', 'camera_la', 'camera_lo', 'drug_S', 'drug_dir', 'file_name', 'height', 'id', 'imgfile',
                'light_color', 'size', 'width']  # 메타데이터 중 같은 이미지의 알약끼리 값이 같은 필드 (나머지는 알약 필드, 00 2-5)
with open(EDA_DIR / '00_raewoo_label_fixes.csv', encoding='utf-8') as f:
    train_fixes = list(csv.DictReader(f))
class_example = {}  # K-코드 → 대회 train에서 그 클래스의 annotation 하나 (그려 넣은 box의 메타데이터용)
for fn in train_files:
    for ann in train_records[fn]['annotations']:
        class_example.setdefault(ann['category_id'], ann)
clean_records = {}
for fn, record in sorted(train_records.items()):
    by_code = {a['category_id']: a for a in record['annotations']}
    anns = [dict(a) for a in record['annotations']]
    for fix in (row for row in train_fixes if row['file_name'] == fn):
        code, box = int(fix['category_id']), json.loads(fix['bbox_after'])
        if not fix['bbox_before']:  # 그려 넣은 box
            example = class_example[code]
            anns.append({**example, 'id': None, 'image_id': record['image']['id'], 'json_path': None, 'bbox': box,
                         'image': {**example['image'], **{key: record['image'][key] for key in IMAGE_FIELDS}}})
            continue
        # 원본 annotation 순서로 찾는다 (라벨을 서로 바꾸면 category_id가 바뀌므로 바뀐 값으로 찾지 않는다)
        ann = anns[next(i for i, a in enumerate(record['annotations']) if a['category_id'] == code)]
        ann['bbox'] = box
        if fix['category_id_after']:  # 라벨이 서로 바뀐 경우: 클래스와 알약 메타데이터를 상대 라벨의 것으로 바꾼다
            other = by_code[int(fix['category_id_after'])]
            ann['category_id'], ann['image'] = other['category_id'], other['image']
    for ann in anns:
        ann['area'] = ann['bbox'][2] * ann['bbox'][3]
    clean_records[fn] = {**record, 'annotations': anns}
clean_object_count = collections.Counter(a['category_id'] for r in clean_records.values() for a in r['annotations'])
assert (len(clean_records), sum(clean_object_count.values()), len(clean_object_count)) == (232, 771, 56), \
    '00 2-5의 정제 결과(이미지 232장, 객체 771개, 클래스 56개)와 다르다. 00_raewoo_label_fixes.csv가 바뀌었는지 확인한다.'

with open(EDA_DIR / '00_raewoo_test_review.csv', encoding='utf-8') as f:
    review_rows = list(csv.DictReader(f))

# AI Hub 라벨 zip (script/download_aihub.py의 폴더 규칙: 라벨 zip은 train_annotations/에 있다)
label_zips = sorted(p for p in (AIHUB_DIR / 'train_annotations').glob('*.zip')
                    if unicodedata.normalize('NFC', p.name).endswith('_조합.zip'))
assert len(label_zips) == 8, (f'허용 조합 라벨 zip 8개가 {AIHUB_DIR / "train_annotations"}에 있어야 한다. 지금 {len(label_zips)}개\n'
                              '받는 명령: uv run python script/download_aihub.py 66065 66067 66068 66069 66070 66071 66072 66243')
aihub_records = {}
for path in label_zips:
    aihub_records.update(read_aihub_label_zip(path))  # 금지 파일이면 여기서 오류가 난다
aihub_png = sorted(p.name for p in AIHUB_IMG_DIR.glob('*.png'))
aihub_index_files = [fn for fn in aihub_png if fn.endswith('_index.png')]  # 90° 사진에 K-코드와 제품명을 적은 참고용 사진 (1-1)
aihub_files = [fn for fn in aihub_png if not fn.endswith('_index.png')]  # 촬영 사진

print(f'대회 train {len(train_files)}장 (정제 라벨 객체 {sum(clean_object_count.values())}개), test {len(test_files)}장, '
      f'test 기록표 {len(review_rows)}줄')
print(f'AI Hub 라벨 zip {len(label_zips)}개: 이미지 {len(aihub_records):,}장, 받은 PNG {len(aihub_png):,}장 (그중 _index.png {len(aihub_index_files)}장)')

- 모든 이미지(대회 train 232장, test 842장, AI Hub)를 한 번씩 읽어, 뒤 분석에 쓸 값을 계산해 둔다.
    - 파일 내용 해시(MD5): 1-6에서 내용이 완전히 같은 파일을 찾는다.
    - 크기(너비, 높이)와 읽기 성공 여부: 1-1에서 해상도와 깨진 파일을 확인한다.
    - 채널별 평균 밝기(R, G, B): 1-4에서 촬영 조건을 비교한다.
    - 32×40 회색 축소 그림: 1-6에서 다시 저장되거나 조금 달라진 같은 사진을 찾는다.
- 이미지가 많아 처음 실행은 몇 분 걸린다. 결과는 `runs/eda_raewoo/cache/image_stats.npz`에 저장하고, 파일 목록이 같으면 다시 계산하지 않는다.

In [ ]:
THUMB_W, THUMB_H = 32, 40  # 축소 그림 크기 (원본 976×1280과 같은 비율)


def image_stats(path):
    """이미지 하나의 MD5, 크기, 채널 평균 밝기, 회색 축소 그림을 계산한다. 읽지 못하면 크기를 (0, 0)으로 둔다."""
    data = path.read_bytes()
    bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        return hashlib.md5(data).hexdigest(), (0, 0), np.zeros(3), np.zeros((THUMB_H, THUMB_W), np.uint8)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    thumb = cv2.resize(gray, (THUMB_W, THUMB_H), interpolation=cv2.INTER_AREA)
    return hashlib.md5(data).hexdigest(), (bgr.shape[1], bgr.shape[0]), bgr.reshape(-1, 3).mean(axis=0)[::-1], thumb


stat_paths = ([('train', TRAIN_IMG_DIR / fn) for fn in train_files] + [('test', TEST_IMG_DIR / fn) for fn in test_files]
              + [('aihub', AIHUB_IMG_DIR / fn) for fn in aihub_png])
stat_keys = np.array([f'{group}/{path.name}' for group, path in stat_paths])
cache_path = CACHE_DIR / 'image_stats.npz'
cache = np.load(cache_path) if cache_path.exists() else None
if cache is not None and np.array_equal(cache['keys'], stat_keys):
    stats = {name: cache[name] for name in cache.files}
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:  # OpenCV는 이미지를 읽는 동안 다른 스레드를 막지 않는다
        results = list(pool.map(image_stats, [path for _, path in stat_paths]))
    stats = {'keys': stat_keys, 'md5': np.array([r[0] for r in results]), 'size': np.array([r[1] for r in results]),
             'mean_rgb': np.array([r[2] for r in results]), 'thumb': np.array([r[3] for r in results])}
    np.savez(cache_path, **stats)
stat_index = {key: i for i, key in enumerate(stats['keys'])}  # 'aihub/<파일명>' → 행 번호
print(f"이미지 {len(stat_keys):,}장의 값을 계산했다. (대회 train {len(train_files)}, test {len(test_files)}, AI Hub {len(aihub_png):,})")

## 1. 데이터셋 훑어보기
- 대회 원본 데이터의 출처는 AI Hub 576 "약품식별 인공지능 개발을 위한 경구약제 이미지 데이터"이다. (DATA.md)
- 대회 규칙은 이 중 `TL_2_조합.zip`, `TS_2_조합.zip` 두 파일만 금지한다. (RULE.md)

### 1-1. 파일 구성과 금지 범위
- `aihubshell -mode l -datasetkey 576`으로 받은 파일 목록을 정리한다. (2026-10-24 조회)
    - TL/VL은 라벨링 데이터(JSON), TS/VS는 원천 데이터(이미지)이다. T는 Training, V는 Validation이다.
    - 용량은 aihubshell이 반올림해 보여 준 값이다.

In [ ]:
AIHUB_FILES = [
    # (구분, 파일, 개수, 파일 하나 용량, filekey, 사용)
    ('Training 조합 라벨', 'TL_1, TL_3~8_조합.zip', 7, '7~8MB', '66065, 66067~66072', '사용 가능'),
    ('Training 조합 라벨', 'TL_2_조합.zip', 1, '7MB', '66066', '**금지**'),
    ('Training 조합 이미지', 'TS_1, TS_3~8_조합.zip', 7, '3GB', '66154, 66156~66161', '사용 가능'),
    ('Training 조합 이미지', 'TS_2_조합.zip', 1, '3GB', '66155', '**금지**'),
    ('Validation 조합 라벨', 'VL_1_조합.zip', 1, '7MB', '66243', '사용 가능'),
    ('Validation 조합 이미지', 'VS_1_조합.zip', 1, '3GB', '66254', '사용 가능'),
    ('Training 단일 라벨', 'TL_1~81_단일.zip', 81, '11~73MB (합계 2.6GB)', '66073~66153', '사용 가능'),
    ('Training 단일 이미지', 'TS_1~81_단일.zip', 81, '13~95GB (합계 3,473GB)', '66162~66242', '사용 가능 (로컬 디스크 부족)'),
    ('Validation 단일 라벨', 'VL_1~10_단일.zip', 10, '15~24MB', '66244~66253', '사용 가능'),
    ('Validation 단일 이미지', 'VS_1~10_단일.zip', 10, '19~31GB (합계 228GB)', '66255~66264', '사용 가능 (로컬 디스크 부족)'),
]
show_table(['구분', '파일', '개수', '용량', 'filekey', '사용'], [list(row) for row in AIHUB_FILES])

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 받은 라벨 zip과 이미지 zip이 서로 맞는지 zip 쌍마다 확인한다.
    - 라벨 zip(TL_n, VL_1)과 같은 번호의 이미지 zip(TS_n, VS_1)에 같은 이미지가 들어 있어야 한다.
    - 이미지가 어느 zip에서 나왔는지는 `script/extract_aihub.py`가 푼 뒤 남긴 `train_images_sources.csv`로 본다. (zip은 풀고 지웠다)
- 이미지마다 실제 PNG 크기가 라벨의 width, height(976×1280)와 같은지, 읽지 못하는 파일이 있는지 본다. (0번에서 계산한 값)
- 이미지 zip에는 촬영 사진(`<조합>_0_2_0_2_<각도>_000_200.png`) 말고 조합마다 `<조합>_index.png`가 하나씩 있다. 라벨이 없으므로 따로 센다.

In [ ]:
with open(AIHUB_SOURCES_CSV, encoding='utf-8') as f:
    image_zip_of = {row['file_name']: row['image_zip'] for row in csv.DictReader(f)}  # PNG 파일명 → 나온 이미지 zip
label_zip_of = {fn: r['zip'] for fn, r in aihub_records.items()}  # 라벨이 있는 이미지 → 라벨 zip
PAIR = {f'{kind}L_{n}_조합.zip': f'{kind}S_{n}_조합.zip' for kind, n in [('T', 1), *[('T', n) for n in range(3, 9)], ('V', 1)]}
aihub_sizes = {fn: tuple(stats['size'][stat_index[f'aihub/{fn}']]) for fn in aihub_png}

rows = []
for label_zip, image_zip in PAIR.items():
    labeled = {fn for fn, z in label_zip_of.items() if z == label_zip}
    images = {fn for fn, z in image_zip_of.items() if z == image_zip and not fn.endswith('_index.png')}
    index = {fn for fn, z in image_zip_of.items() if z == image_zip and fn.endswith('_index.png')}
    on_disk = (images | index) & set(aihub_sizes)
    rows.append([f'{label_zip} / {image_zip}', f'{len(labeled):,}', f'{len(images):,}', len(index), len(labeled - images), len(images - labeled),
                 sum(aihub_sizes[fn] == (0, 0) for fn in on_disk),
                 sum(aihub_sizes[fn] not in ((0, 0), (IMG_W, IMG_H)) for fn in on_disk)])
rows.append(['**합계**', *[f'**{sum(int(str(r[i]).replace(",", "")) for r in rows):,}**' for i in range(1, 8)]])
show_table(['라벨 zip / 이미지 zip', '라벨이 있는 이미지', '촬영 사진 PNG', '_index.png', '라벨만 있음', '사진만 있음', '읽지 못한 PNG',
            f'{IMG_W}×{IMG_H}이 아닌 PNG'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- `_index.png`가 무엇인지, 같은 조합의 90° 촬영 사진과 나란히 놓고 본다.
    - 왼쪽은 90° 촬영 사진, 가운데는 `_index.png`, 오른쪽은 `_index.png`에서 알약 하나 옆을 확대한 것이다.
    - 두 사진의 픽셀 차이(R, G, B 차이의 평균)도 제목에 적는다.

In [ ]:
index_example = sorted(fn for fn in aihub_index_files
                       if not aihub_records.get(fn.replace('_index.png', '_0_2_0_2_90_000_200.png'), {'errors': ['없음']})['errors'])[0]
photo_example = index_example.replace('_index.png', '_0_2_0_2_90_000_200.png')
index_image, photo_image = Image.open(AIHUB_IMG_DIR / index_example).convert('RGB'), Image.open(AIHUB_IMG_DIR / photo_example).convert('RGB')
pixel_diff = np.abs(np.asarray(index_image, float) - np.asarray(photo_image, float)).mean()
ann = aihub_records[photo_example]['annotations'][0]
x, y, w, h = ann['bbox']
zoom = (max(0, x - 40), max(0, y - 40), min(IMG_W, x + w + 260), min(IMG_H, y + h + 60))
fig, axes = plt.subplots(1, 3, figsize=(20, 8), layout='constrained', gridspec_kw={'width_ratios': [1, 1, 1.3]})
axes[0].imshow(photo_image)
axes[0].set_title(f'90° 촬영 사진\n{photo_example}', fontsize=10)
axes[1].imshow(index_image)
axes[1].set_title(f'_index.png (90° 사진과 픽셀 차이 평균 {pixel_diff:.2f})\n{index_example}', fontsize=10)
axes[2].imshow(index_image.crop(zoom))
axes[2].set_title(f'_index.png 확대 (K-{ann["category_id"]:06d} 주변)', fontsize=10)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('이미지 zip의 _index.png와 같은 조합의 90° 촬영 사진', fontsize=14)
save_fig(fig, '1-1b_index_png')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 1-2. 라벨 구조 비교
- 라벨 zip을 그대로 푼 `aihub_raw/train_annotations/`의 JSON을 읽어, Kaggle train 라벨과 같은 항목을 비교한다.
    - dl_idx는 알약 식별자 필드이고, drug_N은 K-코드(`K-016548`)이다.
    - macOS가 zip에 넣은 `._` 파일은 `script/extract_aihub.py`가 풀지 않으므로, 개수는 zip 안의 파일 목록에서 센다.
- JSON 키 구성은 최상위, images, annotations, categories 각각의 키 이름을 순서대로 모아 Kaggle과 같은지 센다.

In [ ]:
def key_structure(data):
    """JSON의 최상위, images, annotations, categories에 있는 키 이름을 순서대로 모은다."""
    return tuple(tuple(part) for part in (data, data['images'][0], data['annotations'][0], data['categories'][0]))


kaggle_json_paths = sorted(TRAIN_ANN_DIR.rglob('*.json'))
kaggle_structures = collections.Counter(key_structure(json.loads(p.read_text(encoding='utf-8'))) for p in kaggle_json_paths)
kaggle_structure = kaggle_structures.most_common(1)[0][0]
raw = collections.Counter()
aihub_ids = {'image': collections.Counter(), 'category': collections.Counter(), 'annotation': collections.Counter(),
             'category_name': collections.Counter()}
raw_json_paths = sorted(AIHUB_RAW_ANN_DIR.rglob('*.json'))
for path in raw_json_paths:
    folder = unicodedata.normalize('NFC', path.parent.name)
    raw['JSON 파일'] += 1
    if ' copy' in path.name:
        raw['copy 중복 파일'] += 1
    if folder.endswith('-오류'):
        raw['-오류 폴더 JSON'] += 1
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        raw['읽을 수 없는 JSON'] += 1
        continue
    image = data['images'][0]
    raw['키 구성 = Kaggle'] += key_structure(data) == kaggle_structure
    k_code = int(image['drug_N'][2:])
    raw['dl_idx = K-코드'] += int(image['dl_idx']) == k_code
    raw['dl_idx = K-코드 - 1'] += int(image['dl_idx']) == k_code - 1
    raw['폴더 이름 = drug_N'] += folder == image['drug_N']
    aihub_ids['image'][image['id']] += 1
    aihub_ids['category_name'][data['categories'][0]['name']] += 1
    for ann in data['annotations']:
        aihub_ids['category'][ann.get('category_id')] += 1
        aihub_ids['annotation'][ann.get('id')] += 1
for path in label_zips:  # ._ 파일은 풀지 않았으므로 zip 목록에서 센다
    with zipfile.ZipFile(path) as zf:
        raw['macOS ._ 파일'] += sum(os.path.basename(n).startswith('._') and n.endswith('.json') for n in zf.namelist())

kaggle_anns = [ann for r in train_records.values() for ann in r['annotations']]
n_valid = raw['JSON 파일'] - raw['읽을 수 없는 JSON']


def top_values(counter):
    """가장 많은 값 두 개를 '값 (개수)' 문자열로 만든다."""
    return ', '.join(f'{v} ({n})' for v, n in counter.most_common(2))


rows = [
    ['폴더 구조', '`<조합>_json/K-<코드>/<이미지명>.json`', '같음 (폴더 이름 = drug_N ' + f"{raw['폴더 이름 = drug_N']}/{n_valid})"],
    ['JSON 키 구성 (최상위, images, annotations, categories)',
     f"한 가지 {kaggle_structures[kaggle_structure]}/{len(kaggle_json_paths)} (키 {', '.join(str(len(p)) for p in kaggle_structure)}개)",
     f"Kaggle과 같음 {raw['키 구성 = Kaggle']}/{n_valid}"],
    ['JSON 파일 수', len(kaggle_json_paths), raw['JSON 파일']],
    ['image id', f'이미지마다 다름 ({len({r["image"]["id"] for r in train_records.values()})}종)', top_values(aihub_ids['image'])],
    ['category_id', f'K-코드 숫자 ({len(train_classes)}종)', top_values(aihub_ids['category'])],
    ['annotation id', f'annotation마다 다름 ({len({a["id"] for a in kaggle_anns})}종)', top_values(aihub_ids['annotation'])],
    ['categories의 name', '제품명 (dl_name)', top_values(aihub_ids['category_name'])],
    ['dl_idx = K-코드', f"{sum(int(a['image']['dl_idx']) == a['category_id'] for a in kaggle_anns)}/{len(kaggle_anns)}",
     f"{raw['dl_idx = K-코드']}/{n_valid}"],
    ['dl_idx = K-코드 - 1', '0', f"{raw['dl_idx = K-코드 - 1']}/{n_valid}"],
    ['-오류 폴더 JSON', '0', raw['-오류 폴더 JSON']],
    ['읽을 수 없는 JSON', '0', raw['읽을 수 없는 JSON']],
    ['copy 중복 파일', '0', raw['copy 중복 파일']],
    ['macOS ._ 파일 (zip 안)', '0', raw['macOS ._ 파일']],
]
show_table(['항목', 'Kaggle train', 'AI Hub 원본 (허용 라벨 zip 8개)'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 같은 알약 K-001900(보령부스파정 5mg)을 같은 각도(70°)에서 찍은 Kaggle JSON 하나와 AI Hub 원본 JSON 하나를 키마다 나란히 놓는다.
    - Kaggle 쪽은 `K-001900-016548-019607-029451_..._70_...` 이미지에 있는 K-001900의 JSON이다. (00 0번의 `train_records` 예시)
    - AI Hub 쪽은 같은 K-코드 폴더의 70° JSON 중 경로 순서로 첫 번째이다. copy 파일과 `-오류` 폴더는 뺀다.
    - 같은 알약이므로 알약 정보는 같아야 하고, 사진이 달라서 생기는 값과 대회 측이 고친 값만 달라야 한다.
- 값이 다른 키는 `비교` 열에 **다름**으로 표시한다. 긴 값은 앞 30자만 보인다.

In [ ]:
EXAMPLE_CODE = 'K-001900'
kaggle_json_path = (TRAIN_ANN_DIR / 'K-001900-016548-019607-029451_json' / EXAMPLE_CODE
                    / 'K-001900-016548-019607-029451_0_2_0_2_70_000_200.json')
kaggle_json = json.loads(kaggle_json_path.read_text(encoding='utf-8'))
aihub_json_path = next(p for p in raw_json_paths if unicodedata.normalize('NFC', p.parent.name) == EXAMPLE_CODE
                       and '_70_' in p.name and ' copy' not in p.name)
aihub_json = json.loads(aihub_json_path.read_text(encoding='utf-8'))


def short(value, n=30):
    """표에 넣을 값을 앞 n자만 남긴다. 빈 문자열은 (빈 값)으로 쓴다."""
    text = str(value).replace('|', '/')
    if text == '':
        return '(빈 값)'
    return text if len(text) <= n else text[:n] + '…'


def compare_json(left, right, left_name, right_name):
    """두 JSON의 키를 하나씩 나란히 놓은 표 행과, 값이 다른 키 이름을 만든다."""
    rows = [['파일', '-', short(left_name, 40), short(right_name, 40), '-'],
            ['최상위', '(키 목록)', ', '.join(left), ', '.join(right), '같음' if list(left) == list(right) else '**다름**']]
    diff = []
    for part in ['images', 'annotations', 'categories']:
        for key, left_value in left[part][0].items():
            right_value = right[part][0].get(key, '(키 없음)')
            if left_value != right_value:
                diff.append(f'{part}.{key}')
            rows.append([part, key, short(left_value), short(right_value), '같음' if left_value == right_value else '**다름**'])
    n_keys = sum(len(left[part][0]) for part in ['images', 'annotations', 'categories'])
    rows.append([f'**값이 다른 키 {len(diff)}/{n_keys}개**', '', '', '', ''])
    return rows, diff


rows, raw_diff = compare_json(kaggle_json, aihub_json, kaggle_json_path.name, aihub_json_path.name)
show_table(['위치', '키', 'Kaggle train', 'AI Hub 원본', '비교'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 1-3. Kaggle 형식으로 바꾼 뒤 다시 비교
- 1-2에서 찾은 차이를 고치는 공용 모듈의 `convert_aihub_to_kaggle`로 AI Hub 라벨을 Kaggle 형식으로 바꾼다.
    - id 3개(image id, annotation id, annotations의 image_id)는 새로 매긴다. image id는 1501부터, annotation id는 5692부터 준다.
        - 대회 train과 test가 image id 1~1500을 나눠 쓰고(test는 파일명 숫자), 대회 train의 annotation id 최댓값이 5691이다.
    - `category_id`, categories `id`, `dl_idx`는 drug_N의 숫자로 바꾸고, categories `name`은 제품명(`dl_name`)으로 바꾼다.
    - area는 너비×높이로 다시 계산한다.
- 라벨에 오류가 있는 이미지도 모두 바꾼다(`skip_errors=False`). 고치거나 뺄지는 2번에서 정한다.
    - bbox가 비었거나 잘못된 알약은 읽을 수 있는 값이 없어 JSON이 생기지 않는다. 이 알약은 2-1에서 "라벨 누락"으로 찾는다.
- 바꾼 JSON은 `aihub_converted/train_annotations/`에 저장한다. 이 셀을 다시 실행하면 폴더를 비우고 새로 만든다.
- 바꾼 JSON을 대회 train과 같은 `load_annotations`로 읽고, 1-2와 같은 항목을 다시 비교한다.

In [ ]:
# 이 폴더는 이 셀이 만드는 결과라서, 예전 실행에서 남은 JSON이 섞이지 않게 비우고 다시 만든다
assert AIHUB_ANN_DIR == EXTERNAL_DIR / 'aihub_converted' / 'train_annotations'
shutil.rmtree(AIHUB_ANN_DIR, ignore_errors=True)
n_converted = convert_aihub_to_kaggle(aihub_records, AIHUB_ANN_DIR, start_image_id=1501, start_annotation_id=5692, skip_errors=False)
aihub_kaggle = load_annotations(AIHUB_ANN_DIR)  # 이미지 파일명 → Kaggle 형식 라벨 (대회 train_records와 같은 구조)
converted_paths = sorted(AIHUB_ANN_DIR.rglob('*.json'))
converted = [json.loads(p.read_text(encoding='utf-8')) for p in converted_paths]
conv_anns = [a for r in aihub_kaggle.values() for a in r['annotations']]

rows = [
    ['바꾼 이미지 / JSON 파일', f'{len(aihub_kaggle):,}장 / {len(converted_paths):,}개',
     f'라벨 zip 이미지 {len(aihub_records):,}장 중 annotation이 하나도 없는 {len(aihub_records) - len(aihub_kaggle)}장은 JSON이 없음'],
    ['JSON 키 구성 = Kaggle', f'{sum(key_structure(d) == kaggle_structure for d in converted):,}/{len(converted):,}', ''],
    ['image id', f"{min(r['image']['id'] for r in aihub_kaggle.values())}~{max(r['image']['id'] for r in aihub_kaggle.values())}",
     f"이미지마다 다름 {len({r['image']['id'] for r in aihub_kaggle.values()}):,}종"],
    ['annotation id', f"{min(a['id'] for a in conv_anns)}~{max(a['id'] for a in conv_anns)}", f"annotation마다 다름 {len({a['id'] for a in conv_anns}):,}종"],
    ['image_id = 그 이미지의 id', f"{sum(a['image_id'] == a['image']['id'] for a in conv_anns):,}/{len(conv_anns):,}", ''],
    ['category_id = drug_N 숫자', f"{sum(a['category_id'] == int(a['image']['drug_N'][2:]) for a in conv_anns):,}/{len(conv_anns):,}", ''],
    ['dl_idx = K-코드', f"{sum(int(a['image']['dl_idx']) == a['category_id'] for a in conv_anns):,}/{len(conv_anns):,}", 'Kaggle과 같음'],
    ['categories의 name = dl_name', f"{sum(d['categories'][0]['name'] == d['images'][0]['dl_name'] for d in converted):,}/{len(converted):,}", ''],
    ['area = 너비×높이', f"{sum(a['area'] == a['bbox'][2] * a['bbox'][3] for a in conv_anns):,}/{len(conv_anns):,}", ''],
    ['클래스 수', f"{len({a['category_id'] for a in conv_anns})}개", f"대회 train {len(train_classes)}개"],
]
show_table(['항목', '바꾼 AI Hub 라벨', '비고'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 1-2의 같은 AI Hub JSON을 바꾼 결과를 같은 Kaggle JSON과 다시 나란히 놓는다.
    - 대회 측 가공과 같아졌다면, 사진이 달라서 생기는 값(파일명, bbox, area)과 이미지마다 새로 매기는 id만 달라야 한다.

In [ ]:
converted_example_path = AIHUB_ANN_DIR / aihub_json_path.parent.parent.name / EXAMPLE_CODE / aihub_json_path.name.replace(' copy', '')
converted_example = json.loads(converted_example_path.read_text(encoding='utf-8'))
rows, converted_diff = compare_json(kaggle_json, converted_example, kaggle_json_path.name, converted_example_path.name)
show_table(['위치', '키', 'Kaggle train', 'AI Hub 바꾼 뒤', '비교'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 1-4. 촬영 조건 비교
- 메타데이터의 촬영 조건 필드를 이미지 단위로 세어 대회 train과 AI Hub를 비교한다.
    - back_color(배경), light_color(조명), camera_la(카메라 위도, 70°/75°/90°), camera_lo(카메라 경도), drug_dir(알약 앞면/뒷면), drug_S(알약 상태), size, width×height
    - 알약마다 JSON이 따로 있으므로, 이미지 하나의 값은 그 이미지 JSON들의 값을 모은 것이다. 알약마다 값이 다르면 `섞임`으로 센다.

In [ ]:
CONDITION_FIELDS = ['back_color', 'light_color', 'camera_la', 'camera_lo', 'drug_dir', 'drug_S', 'size']


def condition_counts(records):
    """촬영 조건 필드마다 이미지 단위 값의 개수를 센다. 같은 이미지의 알약끼리 값이 다르면 '섞임'으로 센다."""
    counts = {field: collections.Counter() for field in CONDITION_FIELDS + ['width×height']}
    for record in records.values():
        for field in CONDITION_FIELDS:
            values = {a['image'][field] for a in record['annotations']} or {record['image'][field]}
            counts[field][values.pop() if len(values) == 1 else '섞임'] += 1
        counts['width×height'][f"{record['image']['width']}×{record['image']['height']}"] += 1
    return counts


def count_text(counter, total):
    """값별 이미지 수를 많은 순서로 '값 n장 (비율)'로 적는다. 셋이 넘으면 나머지는 합친다."""
    items = counter.most_common()
    text = [f'{v} {n:,}장 ({n / total:.1%})' for v, n in items[:3]]
    if len(items) > 3:
        text.append(f'그 외 {sum(n for _, n in items[3:]):,}장')
    return ', '.join(text)


train_conditions = condition_counts(train_records)
aihub_conditions = condition_counts(aihub_kaggle)
show_table(['필드', f'대회 train ({len(train_records)}장)', f'AI Hub ({len(aihub_kaggle):,}장)'],
           [[field, count_text(train_conditions[field], len(train_records)), count_text(aihub_conditions[field], len(aihub_kaggle))]
            for field in CONDITION_FIELDS + ['width×height']])

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 이미지마다 R, G, B 채널의 평균 밝기(0~255)를 대회 train, test, AI Hub로 나눠 분포를 비교한다. (0번에서 계산한 값)
    - 세 묶음의 이미지 수가 크게 달라서(232장, 842장, 약 12,000장) 넓이가 1인 밀도로 그린다.
    - 00 1-2에서 대회 train과 test의 채널 평균 차이는 3~7%였다.

In [ ]:
groups = {'대회 train': 'train', 'test': 'test', 'AI Hub': 'aihub'}
group_means = {name: stats['mean_rgb'][[k.startswith(prefix + '/') and not k.endswith('_index.png') for k in stats['keys']]]
               for name, prefix in groups.items()}
fig, axes = plt.subplots(1, 3, figsize=(20, 5), layout='constrained', sharey=True)
bins = np.linspace(min(m.min() for m in group_means.values()) - 2, max(m.max() for m in group_means.values()) + 2, 60)
for c, (ax, channel) in enumerate(zip(axes, ['R', 'G', 'B'])):
    for (name, means), color in zip(group_means.items(), ['tab:blue', 'tab:orange', 'tab:green']):
        values = means[:, c]
        ax.hist(values, bins=bins, density=True, histtype='step', linewidth=2, color=color,
                label=f'{name} {len(values):,}장 (중앙값 {np.median(values):.1f})')
        ax.axvline(np.median(values), color=color, linestyle='--', linewidth=1)
    ax.set_xlabel(f'{channel} 채널 평균 밝기 (0~255)')
    ax.set_title(f'{channel} 채널')
    ax.legend(fontsize=9)
axes[0].set_ylabel('밀도 (넓이 합 = 1)')
fig.suptitle('이미지별 채널 평균 밝기 분포 (점선 = 중앙값, 분포가 겹칠수록 촬영 조건이 비슷함)', fontsize=14)
save_fig(fig, '1-4_channel_means')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 묶음별(대회 train, test, AI Hub 이미지 zip 8개) 채널 평균 밝기의 중앙값을 표로 비교한다.
    - 밝기는 R, G, B 평균의 합을 3으로 나눈 값(0~255)이다. 대회 test와의 차이를 비율로 함께 적는다.

In [ ]:
def brightness_row(name, means):
    med = np.median(means, axis=0)
    gray = means.mean(axis=1)
    return [name, f'{len(means):,}장', *[f'{v:.1f}' for v in med], f'{np.median(gray):.1f}',
            f'{(np.median(gray) / np.median(test_gray) - 1):+.1%}', f'{np.percentile(gray, 10):.1f} ~ {np.percentile(gray, 90):.1f}']


test_gray = group_means['test'].mean(axis=1)
rows = [brightness_row('대회 train', group_means['대회 train']), brightness_row('test', group_means['test'])]
for image_zip in PAIR.values():
    idx = [stat_index[f'aihub/{fn}'] for fn, z in sorted(image_zip_of.items()) if z == image_zip and not fn.endswith('_index.png')]
    if idx:
        rows.append(brightness_row(f'AI Hub {image_zip}', stats['mean_rgb'][idx]))
show_table(['묶음', '이미지', 'R 중앙값', 'G 중앙값', 'B 중앙값', '밝기 중앙값', 'test 대비', '밝기 10%~90% 범위'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 1-5. 이미지 육안 확인
- AI Hub 이미지 zip 8개에서 1장씩, 라벨 오류가 없는 이미지를 무작위로 골라 바꾼 box(1-3)를 겹쳐 그린다. (seed 고정)
    - box 위 글자는 K-코드와 메타데이터의 각인이다. 알약에 찍힌 글자와 대조하면 box가 맞는 알약을 감싸는지 알 수 있다.
- box가 알약을 빈틈없이 감싸는지, 대회 사진(00 1-3)과 같은 장면인지 본다.

In [ ]:
rng = np.random.default_rng(SEED)
look_files = []
for image_zip in PAIR.values():
    pool = sorted(fn for fn, z in image_zip_of.items() if z == image_zip and fn in aihub_kaggle and not aihub_records[fn]['errors'])
    if pool:  # 아직 받지 않은 zip은 건너뛴다
        look_files.append(pool[rng.integers(len(pool))])

fig, axes = plt.subplots(2, 4, figsize=(24, 18), layout='constrained')
for ax, fn in zip(axes.flat, look_files):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in aihub_kaggle[fn]['annotations']:
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2.5))
        ax.text(x, y - 8, pill_label(ann), fontsize=9, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f'{image_zip_of[fn]}\n{fn}', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('AI Hub 이미지 zip마다 무작위 1장 (연두 box = 바꾼 라벨의 box, 글자 = K-코드(각인))', fontsize=14)
save_fig(fig, '1-5_aihub_samples')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 1-6. 누수 점검
- AI Hub 사진 중에 대회 train, test와 같은 사진이 있는지 세 가지 방법으로 찾는다.
    - 파일 이름: 대회 train 이미지, train 조합, 라벨이 누락된 train 8장(00 2-1)의 이름이 AI Hub에 있는지 센다.
    - 파일 내용: MD5 해시(파일 내용이 1바이트라도 다르면 값이 바뀜)가 같은 파일을 찾는다.
    - 축소 그림: 대회 측이 다시 저장해 파일 내용이 달라도 같은 사진이면, 32×40 회색 축소 그림이 거의 같다.
        - 두 축소 그림의 RMS 차이(픽셀 밝기 차이 제곱의 평균에 제곱근, 0~255)를 계산한다. 0이면 똑같고, 같은 사진을 다시 저장하면 1~2 정도이다.
        - 사진마다 가장 가까운 다른 사진을 찾고, RMS 차이가 작은 순서로 아래 그림에서 눈으로 확인한다.
- AI Hub끼리도 다른 조합에 같은 사진이 있는지 본다. 같은 조합의 각도별 사진은 원래 비슷하므로 뺀다.
    - 다른 조합에 같은 사진이 있으면 학습과 검증에 같은 사진이 들어갈 수 있다.
- `_index.png`는 90° 사진에 글자를 적은 것이라(1-1) 모든 비교에서 뺀다.

In [ ]:
def thumbs(prefix):
    """0번에서 계산한 회색 축소 그림을 묶음별로 꺼낸다. _index.png는 뺀다. (파일명 목록, float32 행렬)"""
    idx = [i for i, k in enumerate(stats['keys']) if k.startswith(prefix + '/') and not k.endswith('_index.png')]
    return [stats['keys'][i].split('/', 1)[1] for i in idx], stats['thumb'][idx].reshape(len(idx), -1).astype(np.float32)


def nearest(a, b, groups=None, chunk=2000):
    """a의 축소 그림마다 b에서 RMS 차이가 가장 작은 그림의 번호와 차이를 찾는다. groups를 주면(a와 b가 같을 때) 같은 묶음은 뺀다."""
    b_sq = (b ** 2).sum(axis=1)
    best_idx, best_rms = np.zeros(len(a), int), np.zeros(len(a))
    for start in range(0, len(a), chunk):
        block = a[start:start + chunk]
        d2 = (block ** 2).sum(axis=1)[:, None] + b_sq[None, :] - 2 * block @ b.T  # 제곱 거리 = |a|² + |b|² - 2a·b
        if groups is not None:
            d2[groups[start:start + chunk, None] == groups[None, :]] = np.inf
        best_idx[start:start + chunk] = d2.argmin(axis=1)
        best_rms[start:start + chunk] = np.sqrt(np.maximum(d2.min(axis=1), 0) / a.shape[1])
    return best_idx, best_rms


train_names, train_thumbs = thumbs('train')
test_names, test_thumbs = thumbs('test')
aihub_names, aihub_thumbs = thumbs('aihub')
near = {
    '대회 train': nearest(train_thumbs, aihub_thumbs),
    'test': nearest(test_thumbs, aihub_thumbs),
    'AI Hub 다른 조합 사진': nearest(aihub_thumbs, aihub_thumbs, groups=np.array([combo_name(fn) for fn in aihub_names])),
}
md5_of = collections.defaultdict(list)
for key, digest in zip(stats['keys'], stats['md5']):
    md5_of[digest].append(key)
same_md5 = {group: sum(any(k.startswith('aihub/') for k in keys) and any(k.startswith(group + '/') for k in keys)
                       for keys in md5_of.values()) for group in ['train', 'test']}
aihub_dup_md5 = sum(sum(k.startswith('aihub/') and not k.endswith('_index.png') for k in keys) > 1 for keys in md5_of.values())
missing_train = [fn for fn in train_files if len({a['category_id'] for a in train_records[fn]['annotations']}) < len(pill_codes(fn))]
aihub_combos = {combo_name(fn) for fn in aihub_records}
NEAR_RMS = 2  # 이 값보다 작으면 같은 사진 후보로 센다 (같은 사진에 글자만 적은 _index.png와 90° 사진이 0.6 정도, 1-1)

rows = [
    ['파일 이름: 대회 train 이미지', f'{len(set(train_files) & set(aihub_records))}/{len(train_files)}'],
    ['파일 이름: 대회 train 조합', f'{len(set(train_combos) & aihub_combos)}/{len(train_combos)}'],
    ['파일 이름: 라벨이 누락된 train 이미지 (00 2-1)', f'{len(set(missing_train) & set(aihub_records))}/{len(missing_train)}'],
    ['파일 내용(MD5)이 같은 사진: AI Hub와 대회 train', same_md5['train']],
    ['파일 내용(MD5)이 같은 사진: AI Hub와 test', same_md5['test']],
    ['파일 내용(MD5)이 같은 사진: AI Hub끼리', aihub_dup_md5],
]
for name, (_, rms) in near.items():
    rows.append([f'축소 그림: {name}에서 가장 가까운 AI Hub 사진의 RMS 차이',
                 f'최소 {rms.min():.1f}, 중앙값 {np.median(rms):.1f}, {NEAR_RMS} 미만 {(rms < NEAR_RMS).sum()}장'])
show_table(['확인 항목', '결과'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 축소 그림이 가장 가까운 쌍을 묶음마다 4쌍씩, RMS 차이가 작은 순서로 나란히 그린다.
    - 윗줄은 대회 train 또는 test 사진(AI Hub끼리는 앞 사진), 아랫줄은 가장 가까운 AI Hub 사진이다.
- 같은 사진인지(알약 배치와 위치가 같은지), 비슷한 장면의 다른 사진인지 본다.

In [ ]:
pair_sources = [('대회 train', train_names, TRAIN_IMG_DIR), ('test', test_names, TEST_IMG_DIR), ('AI Hub 다른 조합 사진', aihub_names, AIHUB_IMG_DIR)]
fig, axes = plt.subplots(2, 12, figsize=(30, 7.5), layout='constrained')
col = 0
for name, names, folder in pair_sources:
    idx, rms = near[name]
    shown = []
    for i in np.argsort(rms):
        pair = frozenset((names[i], aihub_names[idx[i]]))
        if name == 'AI Hub 다른 조합 사진' and pair in shown:  # 같은 쌍이 순서만 바뀌어 두 번 나오는 것을 뺀다
            continue
        shown.append(pair)
        axes[0, col].imshow(Image.open(folder / names[i]))
        axes[0, col].set_title(f'{name}\n{names[i][:34]}', fontsize=8)
        axes[1, col].imshow(Image.open(AIHUB_IMG_DIR / aihub_names[idx[i]]))
        axes[1, col].set_title(f'AI Hub (RMS 차이 {rms[i]:.1f})\n{aihub_names[idx[i]][:34]}', fontsize=8)
        col += 1
        if len(shown) == 4:
            break
for ax in axes.flat:
    ax.axis('off')
fig.suptitle('축소 그림이 가장 가까운 사진 쌍 (윗줄 = 찾는 사진, 아랫줄 = 가장 가까운 AI Hub 사진, RMS 차이가 작은 순서)', fontsize=14)
save_fig(fig, '1-6_nearest_pairs')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

## 2. 라벨 품질
- 00 2절과 같은 항목을 1-3에서 바꾼 AI Hub 라벨(`aihub_kaggle`)로 본다.
- 목적: 고치거나 뺄 라벨과 이미지를 정한다.
    - 평가 지표 mAP@[0.75:0.95]는 IoU 0.75 미만인 box를 오답으로 처리한다. 라벨 box가 조금만 틀려도 학습과 평가가 모두 흔들린다.

### 2-1. 라벨 누락과 클래스 불일치
- 이미지마다 파일명에 적힌 K-코드와 바꾼 라벨의 category_id를 대조한다.
    - 라벨 누락: 파일명에는 있는데 bbox가 없는 K-코드이다.
    - 클래스 불일치: bbox에는 붙어 있는데 파일명에 없는 K-코드이다. 같은 K-코드가 파일명보다 많이 붙은 경우도 센다.
- 빠진 알약마다 원본 JSON(`aihub_raw`)을 찾아 원인을 나눈다.
    - JSON 없음: 그 알약의 JSON 파일이 없다.
    - bbox 빈 값: JSON은 있는데 bbox가 `[]`이다.
    - 숫자가 붙은 bbox: bbox 값이 4개가 아니다. (예: `[121125, 333, 475]`)
    - `-오류` 폴더: JSON이 `K-<코드>-오류` 폴더에 있다.
    - 읽을 수 없는 JSON: JSON 문법이 깨져 읽을 수 없다.

In [ ]:
def raw_label(fn, code):
    """빠진 알약의 원본 JSON을 찾아 (원인, JSON 경로, 원본 bbox)를 돌려준다."""
    folder = AIHUB_RAW_ANN_DIR / f'{combo_name(fn)}_json'
    stem = fn[:-len('.png')]
    error_paths = sorted(folder.glob(f'K-{code:06d}-*/{stem}*.json'))
    paths = sorted((folder / f'K-{code:06d}').glob(f'{stem}*.json'), key=lambda p: (' copy' in p.name, p.name))
    if error_paths:
        paths, cause = error_paths, '-오류 폴더'
    elif not paths:
        return 'JSON 없음', None, None
    try:
        anns = json.loads(paths[0].read_text(encoding='utf-8'))['annotations']
    except json.JSONDecodeError:
        return '읽을 수 없는 JSON', paths[0], None
    bbox = anns[0].get('bbox') if anns else None
    if error_paths:
        return cause, paths[0], bbox
    if bbox == []:
        return 'bbox 빈 값', paths[0], bbox
    if not isinstance(bbox, list) or len(bbox) != 4:
        return '숫자가 붙은 bbox', paths[0], bbox
    return 'bbox 너비·높이 0 이하', paths[0], bbox


missing_labels = []  # (이미지, 빠진 K-코드, 원인, 원본 JSON 경로, 원본 bbox)
extra_labels = []  # (이미지, 파일명에 없는 K-코드)
for fn in sorted(aihub_records):
    named = collections.Counter(pill_codes(fn))
    labeled = collections.Counter(a['category_id'] for a in aihub_kaggle.get(fn, {'annotations': []})['annotations'])
    for code in sorted((named - labeled).elements()):
        missing_labels.append((fn, code, *raw_label(fn, code)))
    extra_labels += [(fn, code) for code in sorted((labeled - named).elements())]
CAUSES = ['JSON 없음', 'bbox 빈 값', '숫자가 붙은 bbox', '-오류 폴더', '읽을 수 없는 JSON', 'bbox 너비·높이 0 이하']
missing_files = sorted({m[0] for m in missing_labels})
per_image = collections.Counter(collections.Counter(m[0] for m in missing_labels).values())

rows = []
for cause in CAUSES:
    items = [m for m in missing_labels if m[2] == cause]
    if not items:
        continue
    by_zip = collections.Counter(label_zip_of[m[0]].replace('_조합.zip', '') for m in items)
    rows.append([cause, len(items), len({m[0] for m in items}), ', '.join(f'{z} {n}' for z, n in sorted(by_zip.items())),
                 len({m[1] for m in items})])
rows.append([f'**라벨 누락 합계**', f'**{len(missing_labels)}**', f'**{len(missing_files)}장**', '', f"**{len({m[1] for m in missing_labels})}**"])
rows.append(['이미지당 빠진 알약 수', '', ', '.join(f'{k}개 {v}장' for k, v in sorted(per_image.items())), '', ''])
rows.append(['클래스 불일치 (파일명에 없는 K-코드)', len(extra_labels), len({e[0] for e in extra_labels}), '', len({e[1] for e in extra_labels})])
show_table(['원인', '빠진 알약 수', '이미지 수', 'zip별 빠진 알약 수', '클래스 수'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 원인마다 예시 이미지를 최대 2장씩 그린다.
    - 연두 box는 라벨에 있는 box이고, 제목 둘째 줄은 빠진 알약의 K-코드와 각인이다.
    - 빠진 알약의 각인은 같은 클래스의 다른 AI Hub annotation에서 가져온다.
- 빠진 알약이 사진에 실제로 있는지, 어느 알약인지 본다.

In [ ]:
aihub_example = {}  # K-코드 → 바꾼 AI Hub 라벨에서 그 클래스의 annotation 하나 (각인 표시용)
for fn in sorted(aihub_kaggle):
    for ann in aihub_kaggle[fn]['annotations']:
        aihub_example.setdefault(ann['category_id'], ann)


def missing_text(fn):
    """그 이미지에서 빠진 알약의 K-코드와 각인을 적는다."""
    return ', '.join(pill_label(aihub_example[m[1]]) if m[1] in aihub_example else f'K-{m[1]:06d}'
                     for m in missing_labels if m[0] == fn)


examples = []
for cause in CAUSES:
    files = sorted({m[0] for m in missing_labels if m[2] == cause})
    examples += [(cause, fn) for fn in files[:2]]
n_cols = 5
fig, axes = plt.subplots(int(np.ceil(len(examples) / n_cols)), n_cols, figsize=(30, 9.5 * np.ceil(len(examples) / n_cols)),
                         layout='constrained', squeeze=False)
for ax, (cause, fn) in zip(axes.flat, examples):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2.5))
        ax.text(x, y - 8, pill_label(ann), fontsize=8, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f'[{cause}] {fn}\nbbox 없음: {missing_text(fn)}', fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    if not ax.images:
        ax.axis('off')
fig.suptitle('라벨이 누락된 이미지의 원인별 예시 (연두 box = 라벨에 있는 box, 제목 둘째 줄 = 파일명에는 있지만 box가 없는 알약)', fontsize=14)
save_fig(fig, '2-1_missing_examples')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 같은 K-코드의 JSON끼리 알약 정보가 같은지 확인한다.
    - 알약 정보 필드: dl_name(제품명), print_front·print_back(각인), drug_shape(모양), color_class1(색), leng_long·leng_short·thick(크기)
    - 같은 알약이면 사진이 달라도 이 값들이 같아야 한다. 다르면 다른 알약의 JSON이 섞였을 수 있다.
- 대회 train과 같은 클래스는 대회 JSON의 값과도 비교한다.

In [ ]:
PILL_FIELDS = ['dl_name', 'print_front', 'print_back', 'drug_shape', 'color_class1', 'leng_long', 'leng_short', 'thick']
aihub_field_values = collections.defaultdict(lambda: collections.defaultdict(collections.Counter))  # K-코드 → 필드 → 값 개수
for record in aihub_kaggle.values():
    for ann in record['annotations']:
        for field in PILL_FIELDS:
            aihub_field_values[ann['category_id']][field][str(ann['image'].get(field))] += 1
kaggle_field_values = {code: {field: str(ann['image'].get(field)) for field in PILL_FIELDS} for code, ann in class_example.items()}

rows = []
for code in sorted(aihub_field_values):
    for field in PILL_FIELDS:
        values = aihub_field_values[code][field]
        kaggle_value = kaggle_field_values.get(code, {}).get(field)
        if len(values) > 1 or (kaggle_value is not None and kaggle_value not in values):
            rows.append([f'K-{code:06d}', field, ', '.join(f'{v} ({n})' for v, n in values.most_common()), kaggle_value or '-'])
n_classes = len(aihub_field_values)
n_bad = len({r[0] for r in rows})
rows.append([f'**{n_classes}개 클래스 중 값이 하나가 아닌 클래스 {n_bad}개**', '', '', ''])
show_table(['K-코드', '필드', 'AI Hub 값 (annotation 수)', '대회 train 값'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 2-2. 좌표 오류와 고치는 방법
- 00 2-2와 같은 오류를 원본 JSON(`aihub_raw`)의 bbox와 area로 찾는다.
    - 이미지 밖: bbox가 이미지(976×1280) 밖으로 나간다.
    - area 불일치: area가 bbox 너비×높이와 다르다. area 값의 오타일 수도 있고, bbox 값의 오타일 수도 있다.
    - 다른 클래스와 box가 똑같음: 같은 이미지 안에서 두 클래스의 box 좌표가 똑같다. 한 알약의 box를 다른 클래스에도 복사한 오류이다.
- 2-1의 라벨 누락(원본 bbox가 없거나 쓸 수 없는 알약)도 같은 표에 함께 센다. 고치는 방법이 같기 때문이다.
- 사진이 있는 이미지만 센다. 1-1의 "라벨만 있음" 이미지는 사진으로 고칠 수 없고, 2-5에서 학습 데이터에서 빠진다.

In [ ]:
def raw_annotation(fn, code):
    """원본 JSON에서 그 알약의 (bbox, area)를 읽는다. JSON이 없거나 읽을 수 없으면 (None, None)이다."""
    paths = sorted((AIHUB_RAW_ANN_DIR / f'{combo_name(fn)}_json' / f'K-{code:06d}').glob(f'{fn[:-4]}*.json'),
                   key=lambda p: (' copy' in p.name, p.name))
    try:
        ann = json.loads(paths[0].read_text(encoding='utf-8'))['annotations'][0]
    except (IndexError, json.JSONDecodeError):
        return None, None
    return ann.get('bbox'), ann.get('area')


problems = collections.defaultdict(list)  # 이미지 → [(K-코드, 문제, 원본 bbox, 원본 area)]
photo_files = set(aihub_files)  # 사진이 없는 이미지(1-1의 "라벨만 있음")는 사진으로 고칠 수 없고 2-5에서 빠지므로 세지 않는다
for fn in sorted(set(aihub_kaggle) & photo_files):
    anns = aihub_kaggle[fn]['annotations']
    same_box = collections.Counter(tuple(a['bbox']) for a in anns)
    for ann in anns:
        x, y, w, h = ann['bbox']
        raw_box, raw_area = raw_annotation(fn, ann['category_id'])
        kinds = []
        if same_box[tuple(ann['bbox'])] > 1:
            kinds.append('다른 클래스와 box가 똑같음')
        if x < 0 or y < 0 or x + w > IMG_W or y + h > IMG_H:
            kinds.append('이미지 밖')
        if raw_area != w * h:
            kinds.append('area 불일치')
        if kinds:
            problems[fn].append((ann['category_id'], ' + '.join(kinds), raw_box, raw_area))
for fn, code, cause, path, raw_box in missing_labels:  # 2-1의 라벨 누락
    if fn not in photo_files:
        continue
    raw_area = raw_annotation(fn, code)[1] if path and cause == '숫자가 붙은 bbox' else None
    problems[fn].append((code, f'라벨 누락: {cause}', raw_box, raw_area))

kind_count = collections.Counter(kind for items in problems.values() for _, kind, _, _ in items)
rows = []
for kind, n in kind_count.most_common():
    files = sorted(fn for fn, items in problems.items() if any(k == kind for _, k, _, _ in items))
    example = next(item for item in problems[files[0]] if item[1] == kind)
    rows.append([kind, n, len(files), f'`{files[0]}` K-{example[0]:06d} bbox {example[2]}, area {example[3]}'])
rows.append(['**합계**', f'**{sum(kind_count.values())}**', f'**{len(problems)}장**', ''])
show_table(['문제', 'annotation 수', '이미지 수', '예시 (이미지, 원본 bbox, 원본 area)'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 문제가 있는 box는 사진에서 알약을 직접 찾아 고친다. 00 2-1은 영역을 눈으로 정했지만, 여기는 이미지가 많아 OpenCV로 알약 자리까지 찾는다.
    - 바닥 추정: 라벨 box 자리를 뺀 나머지 픽셀을 크게 흐려, 위치마다 바닥의 밝기(R+G+B)와 파랑 비율 B/(R+G+B)를 구한다. 사진 위쪽이 아래쪽보다 밝기 때문이다.
    - 알약 픽셀: 00 2-1의 기준(바닥보다 파랑 비율이 0.04 이상 낮거나, 20% 이상 밝음)에, 바닥보다 15% 이상 어둡고 더 푸르지 않은 픽셀을 더한다.
        - 검은 캡슐은 바닥보다 어둡지만 색 비율은 바닥과 같아서 00의 기준으로는 찾지 못한다. 그림자는 바닥보다 어둡고 더 푸르러서 빠진다.
    - 알약 덩어리: 1/4로 줄인 사진에서 알약 픽셀이 붙은 덩어리(`cv2.connectedComponentsWithStats`)를 찾는다. 작은 잡음은 뺀다.
        - 조명 밖의 밝은 모서리는 밝기 기준에 걸리므로, 밝기로만 찾은 픽셀 중 사진 테두리와 이어진 덩어리를 먼저 지운다. 테두리 가까이 놓인 색 있는 알약은 남는다.
    - 알약 범위: 덩어리 주변을 원본 해상도로 다시 보고, 00 2-1처럼 알약 픽셀이 충분한 행과 열의 범위를 잰다.
    - box: 알약 범위의 네 변에 클래스별 여백(라벨 box 변과 알약 범위 사이 거리)의 중앙값을 더한다.
- 클래스 기준값: 라벨에 문제가 없는 이미지 3,000장(seed 고정)에서 라벨 box마다 알약 범위를 재서, 클래스마다 여백, 알약 범위의 긴 변·짧은 변, 가운데 색(Lab)의 중앙값을 구한다.
    - Lab은 사람 눈이 느끼는 색 차이에 가깝게 만든 색 공간이다. 찾은 알약이 어느 클래스인지 크기와 색으로 정할 때 쓴다.
- 검증: 기준값을 만들 때 쓰지 않은 다른 이미지 1,000장으로 확인한다.
    - box 하나 지우기: 라벨 box 하나를 지우고 사진에서 다시 찾아, 크기와 색이 그 클래스에 가장 가까운 덩어리로 만든 box와 라벨 box의 IoU를 본다.
    - box 둘 지우기: 라벨 box 둘을 지우고 다시 찾아, 크기와 색으로 두 클래스를 맞는 알약에 배정하는지 본다. (box가 똑같은 오류와 알약 여러 개가 빠진 이미지에 쓴다)
- 계산에 몇 분 걸려 결과를 `runs/eda_raewoo/cache/2-2_pill_finder.json`에 저장한다.

In [ ]:
SCALE = 4  # 알약 덩어리를 찾을 때 사진을 줄이는 배율
BLUE_DROP = 0.04  # 바닥보다 파랑 비율이 이만큼 낮으면 알약 픽셀 (00 2-1)
BRIGHT_RISE = 0.2  # 바닥보다 이 비율 이상 밝으면 알약 픽셀 (흰 알약, 00 2-1)
DARK_DROP, DARK_BLUE = 0.15, 0.01  # 바닥보다 15% 이상 어둡고 파랑 비율이 바닥+0.01보다 낮으면 알약 픽셀 (검은 캡슐)
MIN_SHARE = 0.1  # 알약 픽셀이 가장 많은 행(열)의 이 비율 이상인 행(열)만 알약 범위에 넣는다 (00 2-1)
REFINE_PAD = 24  # 덩어리 주변을 원본 해상도로 다시 볼 때 넓히는 폭 (픽셀)
MIN_COMPONENT = 300  # 줄인 사진에서 이보다 작은 덩어리(원본 약 4,800픽셀)는 알약으로 보지 않는다
N_REF, N_VAL = 3000, 1000  # 클래스 기준값과 검증에 쓰는 이미지 수


def read_rgb(path):
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)


def box_area_mask(shape, boxes, scale=1, pad=0):
    """xywh box 자리를 True로 칠한 마스크를 만든다."""
    mask = np.zeros(shape, bool)
    for x, y, w, h in boxes:
        mask[max(0, int(y) // scale - pad):int(y + h) // scale + pad, max(0, int(x) // scale - pad):int(x + w) // scale + pad] = True
    return mask


def background(rgb, known_boxes):
    """위치마다 바닥의 밝기(R+G+B)와 파랑 비율을 추정한다. 라벨 box 자리와 바닥과 크게 다른 픽셀은 빼고 흐리게 평균낸다."""
    small = cv2.resize(rgb, (IMG_W // SCALE, IMG_H // SCALE), interpolation=cv2.INTER_AREA).astype(np.float32)
    total = small.sum(axis=2) + 1
    blue = small[..., 2] / total
    known = box_area_mask(total.shape, known_boxes, SCALE, 4)
    floor = (~known) & (np.abs(blue - np.median(blue[~known])) < 0.03) & (np.abs(total / np.median(total[~known]) - 1) < 0.15)
    weight = np.maximum(cv2.GaussianBlur(floor.astype(np.float32), (0, 0), 25), 1e-3)
    bg_total = cv2.GaussianBlur(np.where(floor, total, 0).astype(np.float32), (0, 0), 25) / weight
    bg_blue = cv2.GaussianBlur(np.where(floor, blue, 0).astype(np.float32), (0, 0), 25) / weight
    return cv2.resize(bg_total, (IMG_W, IMG_H)), cv2.resize(bg_blue, (IMG_W, IMG_H))


def pill_pixels(rgb, bg_total, bg_blue, split=False):
    """바닥과 다른 픽셀(알약)을 True로 표시한다. split이면 (색으로 찾은 픽셀, 밝기로만 찾은 픽셀)을 따로 돌려준다."""
    rgb = rgb.astype(np.float32)
    total = rgb.sum(axis=2) + 1
    blue = rgb[..., 2] / total
    colored = (blue < bg_blue - BLUE_DROP) | ((total < bg_total * (1 - DARK_DROP)) & (blue < bg_blue + DARK_BLUE))
    bright = (total > bg_total * (1 + BRIGHT_RISE)) & ~colored
    return (colored, bright) if split else (colored | bright)


def corner_zone(bright):
    """밝기로만 찾은 픽셀 중 사진 테두리와 이어진 덩어리(조명 밖의 밝은 모서리)를 찾는다."""
    _, labels = cv2.connectedComponents(bright.astype(np.uint8), connectivity=8)
    edge = np.unique(np.concatenate([labels[0], labels[-1], labels[:, 0], labels[:, -1]]))
    return np.isin(labels, edge[edge > 0])


def refine_extent(rgb, box, bg_total, bg_blue, known_boxes, corner=None):
    """box 주변을 원본 해상도로 보고 가장 큰 알약 덩어리의 범위 [x0, y0, x1, y1]를 찾는다. 다른 라벨 box 자리와 밝은 모서리는 뺀다."""
    x, y, w, h = box
    x0, y0 = max(0, int(x) - REFINE_PAD), max(0, int(y) - REFINE_PAD)
    x1, y1 = min(IMG_W, int(x + w) + REFINE_PAD), min(IMG_H, int(y + h) + REFINE_PAD)
    if x1 - x0 < 8 or y1 - y0 < 8:
        return None  # box가 이미지 밖에 있다
    mask = pill_pixels(rgb[y0:y1, x0:x1], bg_total[y0:y1, x0:x1], bg_blue[y0:y1, x0:x1])
    if corner is not None:
        mask &= ~corner[y0:y1, x0:x1]
    for kx, ky, kw, kh in known_boxes:
        mask[max(0, int(ky) - y0):max(0, int(ky + kh) - y0), max(0, int(kx) - x0):max(0, int(kx + kw) - x0)] = False
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n, labels, comp_stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None
    comp = labels == 1 + int(np.argmax(comp_stats[1:, 4]))
    rows = np.nonzero(comp.sum(axis=1) >= MIN_SHARE * comp.sum(axis=1).max())[0]
    cols = np.nonzero(comp.sum(axis=0) >= MIN_SHARE * comp.sum(axis=0).max())[0]
    return [x0 + int(cols[0]), y0 + int(rows[0]), x0 + int(cols[-1]) + 1, y0 + int(rows[-1]) + 1]


def find_pills(rgb, known_boxes):
    """라벨 box 밖에서 알약을 찾아 알약 범위 [x0, y0, x1, y1] 목록을 큰 덩어리부터 돌려준다."""
    bg_total, bg_blue = background(rgb, known_boxes)
    small = cv2.resize(rgb, (IMG_W // SCALE, IMG_H // SCALE), interpolation=cv2.INTER_AREA)
    size = small.shape[1::-1]
    colored, bright = pill_pixels(small, cv2.resize(bg_total, size), cv2.resize(bg_blue, size), split=True)
    corner = cv2.dilate(corner_zone(bright).astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    mask = (colored | bright) & ~corner & ~box_area_mask(colored.shape, known_boxes, SCALE, 4)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    _, _, comp_stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    corner_full = cv2.resize(corner.astype(np.uint8), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST).astype(bool)
    extents = []
    for x, y, w, h, area in sorted(comp_stats[1:], key=lambda s: -s[4]):
        if area < MIN_COMPONENT:
            continue
        extent = refine_extent(rgb, [x * SCALE, y * SCALE, w * SCALE, h * SCALE], bg_total, bg_blue, known_boxes, corner_full)
        if extent:
            extents.append(extent)
    return extents


def extent_color(rgb, extent):
    """알약 범위 가운데 절반의 평균 색을 Lab으로 구한다."""
    x0, y0, x1, y1 = extent
    patch = rgb[y0 + (y1 - y0) // 4:y1 - (y1 - y0) // 4, x0 + (x1 - x0) // 4:x1 - (x1 - x0) // 4]
    return cv2.cvtColor(patch, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(float).mean(axis=0)


def appearance_cost(extent, lab, ref):
    """알약 범위의 긴 변·짧은 변과 Lab 색이 클래스 기준값과 얼마나 다른지 잰다. 0이면 같다. (색 차이 30이 크기 e배 차이와 같은 무게)"""
    long_side, short_side = sorted([extent[2] - extent[0], extent[3] - extent[1]], reverse=True)
    return (abs(np.log(long_side / ref['long'])) + abs(np.log(short_side / ref['short']))
            + np.linalg.norm(np.asarray(lab) - np.asarray(ref['lab'])) / 30)


def with_margin(extent, margin):
    """알약 범위에 여백(왼, 위, 오른, 아래)을 더해 xywh box를 만든다. 이미지 밖으로 나가지 않게 자른다."""
    x0, y0, x1, y1 = extent
    x, y = max(0, round(x0 - margin[0])), max(0, round(y0 - margin[1]))
    return [int(x), int(y), int(min(IMG_W, round(x1 + margin[2])) - x), int(min(IMG_H, round(y1 + margin[3])) - y)]


finder_cache = CACHE_DIR / '2-2_pill_finder.json'
clean_pool = sorted(fn for fn in aihub_kaggle if fn not in problems and fn in set(aihub_files))  # 라벨에 문제가 없는 이미지
order = np.random.default_rng(SEED).permutation(len(clean_pool))
ref_files, val_files = [clean_pool[i] for i in order[:N_REF]], [clean_pool[i] for i in order[N_REF:N_REF + N_VAL]]
if finder_cache.exists() and json.loads(finder_cache.read_text())['files'] == [ref_files, val_files]:
    finder = json.loads(finder_cache.read_text())
else:
    measures = collections.defaultdict(list)  # K-코드 → [여백 4개, 긴 변, 짧은 변, Lab 3개]
    for fn in ref_files:
        rgb = read_rgb(AIHUB_IMG_DIR / fn)
        boxes = [a['bbox'] for a in aihub_kaggle[fn]['annotations']]
        bg_total, bg_blue = background(rgb, boxes)
        for k, ann in enumerate(aihub_kaggle[fn]['annotations']):
            extent = refine_extent(rgb, ann['bbox'], bg_total, bg_blue, boxes[:k] + boxes[k + 1:])
            if extent:
                x, y, w, h = ann['bbox']
                measures[ann['category_id']].append([extent[0] - x, extent[1] - y, x + w - extent[2], y + h - extent[3],
                                                     *sorted([extent[2] - extent[0], extent[3] - extent[1]], reverse=True),
                                                     *extent_color(rgb, extent)])
    global_margin = np.median([m[:4] for ms in measures.values() for m in ms], axis=0)
    refs = {str(c): {'margin': (np.median(np.array(ms)[:, :4], axis=0) if len(ms) >= 3 else global_margin).tolist(),
                     'long': float(np.median(np.array(ms)[:, 4])), 'short': float(np.median(np.array(ms)[:, 5])),
                     'lab': np.median(np.array(ms)[:, 6:9], axis=0).tolist(), 'n': len(ms)} for c, ms in measures.items()}
    hide_one, hide_two = [], []  # (K-코드, IoU), (맞게 배정했는지)
    rng = np.random.default_rng(SEED)
    for fn in val_files:
        rgb = read_rgb(AIHUB_IMG_DIR / fn)
        anns = [a for a in aihub_kaggle[fn]['annotations']]
        boxes = [a['bbox'] for a in anns]
        for k, ann in enumerate(anns):
            ref = refs.get(str(ann['category_id']))
            extents = find_pills(rgb, boxes[:k] + boxes[k + 1:]) if ref else []
            if not ref:
                continue
            if not extents:
                hide_one.append((ann['category_id'], 0.0))
                continue
            best = min(extents, key=lambda e: appearance_cost(e, extent_color(rgb, e), ref))
            hide_one.append((ann['category_id'], box_iou(with_margin(best, ref['margin']), ann['bbox'])))
        pair = [k for k in rng.permutation(len(anns))[:2] if str(anns[k]['category_id']) in refs]
        if len(pair) == 2:
            extents = find_pills(rgb, [b for k, b in enumerate(boxes) if k not in pair])
            if len(extents) < 2:
                hide_two += [False, False]
                continue
            cost = np.array([[appearance_cost(e, extent_color(rgb, e), refs[str(anns[k]['category_id'])]) for e in extents] for k in pair])
            for i, j in zip(*scipy.optimize.linear_sum_assignment(cost)):
                ref = refs[str(anns[pair[i]]['category_id'])]
                hide_two.append(box_iou(with_margin(extents[j], ref['margin']), anns[pair[i]]['bbox']) > 0.5)
    finder = {'files': [ref_files, val_files], 'refs': refs, 'global_margin': global_margin.tolist(),
              'hide_one': hide_one, 'hide_two': [bool(v) for v in hide_two]}
    finder_cache.write_text(json.dumps(finder))
class_refs = {int(c): ref for c, ref in finder['refs'].items()}
hide_one_iou = np.array([v for _, v in finder['hide_one']])

fig, axes = plt.subplots(1, 2, figsize=(20, 5.5), layout='constrained', gridspec_kw={'width_ratios': [1, 2]})
axes[0].hist(hide_one_iou, bins=np.linspace(0, 1, 51), color='tab:blue')
for t in [0.75, 0.95]:
    axes[0].axvline(t, color='gray', linestyle='--', linewidth=1)
axes[0].set_yscale('log')
axes[0].set_xlabel('다시 찾은 box와 라벨 box의 IoU')
axes[0].set_ylabel('box 수 (로그 눈금)')
axes[0].set_title(f'box 하나 지우고 다시 찾기 {len(hide_one_iou):,}개: 중앙값 {np.median(hide_one_iou):.3f}, '
                  f'0.75 미만 {(hide_one_iou < 0.75).mean():.1%}, 0.95 이상 {(hide_one_iou >= 0.95).mean():.1%}', fontsize=11)
by_class = collections.defaultdict(list)
for c, v in finder['hide_one']:
    by_class[c].append(v)
order = sorted(by_class, key=lambda c: np.median(by_class[c]))
axes[1].bar(np.arange(len(order)), [np.median(by_class[c]) for c in order], color='tab:blue')
axes[1].axhline(0.95, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(0.75, color='gray', linestyle='--', linewidth=1)
axes[1].set_xticks(np.arange(len(order)), [f'K-{c:06d}' for c in order], rotation=90, fontsize=6)
axes[1].set_ylim(0.5, 1)
axes[1].set_ylabel('IoU 중앙값')
axes[1].set_title(f'클래스별 IoU 중앙값 ({len(order)}개 클래스, 낮은 순서)', fontsize=11)
fig.suptitle(f'OpenCV로 빠진 box를 찾는 방법의 검증 (이미지 {N_VAL:,}장, 회색 점선 = IoU 0.75와 0.95), '
             f"box 둘 지우고 클래스 배정: {sum(finder['hide_two'])}/{len(finder['hide_two'])} 맞음", fontsize=13)
save_fig(fig, '2-2_finder_validation')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 좌표 값은 멀쩡해도 box가 알약을 감싸지 않는 경우가 있다. (box가 바닥이나 옆 알약을 감쌈, 한쪽으로 밀림)
    - 이런 box는 2-2 표의 검사로는 찾을 수 없어서, 모든 box를 사진과 대조한다.
- box 맞음: 라벨 box 주변에서 알약 범위를 재고(`refine_extent`), 클래스 여백을 더해 다시 만든 box와 라벨 box의 IoU이다.
    - 1이면 라벨 box가 알약을 클래스의 다른 box들과 같은 여백으로 감싼다. 0이면 box 안에 알약이 없거나 다른 자리에 있다.
    - 0.75 미만인 box를 문제("알약을 감싸지 않는 box")로 보고, 2-2 표의 문제와 함께 아래 규칙으로 고친다.
        - 0.75는 평가 지표의 가장 낮은 IoU 기준이다. 위 검증에서 box 하나 지우기의 IoU가 0.75 미만인 경우는 1~2%였다.
    - 알약 범위를 잘 재지 못하는 클래스는 이 검사에서 뺀다. 이 클래스는 갤러리(2-3)로 본다.
        - 위 검증에서 IoU 중앙값이 0.85 미만인 클래스
        - box 맞음 0.75 미만이 그 클래스 box의 30%를 넘는 클래스 (라벨이 그만큼 틀렸을 가능성보다 재는 방법이 실패했을 가능성이 크다)
- 여기서 잰 알약 범위와 색은 2-3의 크기·색 대조에도 쓴다. 결과는 `runs/eda_raewoo/cache/2-2_box_fit.json`에 저장한다.

In [ ]:
FIT_THRESHOLD = 0.75


def measure_boxes(job):
    """이미지 하나의 box마다 (알약 범위, Lab 색)을 잰다. 알약 범위를 못 찾으면 None이다."""
    fn, boxes = job
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    bg_total, bg_blue = background(rgb, boxes)
    out = []
    for k, box in enumerate(boxes):
        extent = refine_extent(rgb, box, bg_total, bg_blue, boxes[:k] + boxes[k + 1:])
        out.append([fn, box, extent, extent_color(rgb, extent).tolist() if extent else None])
    return out


def measure_all(records):
    """box마다 알약 범위와 색을 잰다. 이미 잰 (이미지, box)는 캐시에서 읽는다."""
    cache_path = CACHE_DIR / '2-2_box_fit.json'
    cached = {(fn, tuple(box)): (extent, lab) for fn, box, extent, lab in (json.loads(cache_path.read_text()) if cache_path.exists() else [])}
    jobs = [(fn, [a['bbox'] for a in r['annotations']]) for fn, r in sorted(records.items())
            if any((fn, tuple(a['bbox'])) not in cached for a in r['annotations'])]
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        for rows in pool.map(measure_boxes, jobs):
            cached.update({(fn, tuple(box)): (extent, lab) for fn, box, extent, lab in rows})
    cache_path.write_text(json.dumps([[fn, list(box), extent, lab] for (fn, box), (extent, lab) in cached.items()]))
    return cached


box_measures = measure_all({fn: r for fn, r in aihub_kaggle.items() if fn in set(aihub_files)})
no_ref = {'margin': finder['global_margin']}
box_fit = {}  # (이미지, K-코드) → box 맞음 IoU
for fn, record in aihub_kaggle.items():
    if fn not in set(aihub_files):
        continue
    for ann in record['annotations']:
        extent, _ = box_measures[(fn, tuple(ann['bbox']))]
        margin = class_refs.get(ann['category_id'], no_ref)['margin']
        box_fit[(fn, ann['category_id'])] = box_iou(with_margin(extent, margin), ann['bbox']) if extent else 0.0
# 위 검증에서 다시 찾기가 잘 안 되는 클래스(IoU 중앙값 0.85 미만, 예: 바닥과 색이 같은 파란 알약)는 box 맞음을 믿을 수 없어 뺀다
validation_by_class = collections.defaultdict(list)
for c, v in finder['hide_one']:
    validation_by_class[c].append(v)
low_share = collections.defaultdict(list)
for (fn, code), v in box_fit.items():
    low_share[code].append(v < FIT_THRESHOLD)
# 한 클래스의 30% 넘게 box 맞음이 낮으면 라벨 오류가 아니라 알약 범위를 못 재는 클래스로 본다
unreliable_classes = sorted({c for c, v in validation_by_class.items() if np.median(v) < 0.85}
                            | {c for c, v in low_share.items() if len(v) >= 10 and np.mean(v) > 0.3})
already = {(fn, code) for fn, items in problems.items() for code, *_ in items}
low_fit = sorted(key for key, v in box_fit.items() if v < FIT_THRESHOLD and key not in already and key[1] not in unreliable_classes)
for fn, code in low_fit:
    raw_box, raw_area = raw_annotation(fn, code)
    problems[fn].append((code, '알약을 감싸지 않는 box', raw_box, raw_area))
kind_count = collections.Counter(kind for items in problems.values() for _, kind, _, _ in items)

fit_values = np.array(list(box_fit.values()))
fig, axes = plt.subplots(1, 2, figsize=(20, 5.5), layout='constrained', gridspec_kw={'width_ratios': [1, 1.6]})
axes[0].hist(fit_values, bins=np.linspace(0, 1, 51), color='tab:blue')
axes[0].axvline(FIT_THRESHOLD, color='red', linestyle='--', linewidth=1)
axes[0].set_yscale('log')
axes[0].set_xlabel('box 맞음 (다시 만든 box와 라벨 box의 IoU)')
axes[0].set_ylabel('box 수 (로그 눈금)')
axes[0].set_title(f'box {len(fit_values):,}개: 중앙값 {np.median(fit_values):.3f}, {FIT_THRESHOLD} 미만 {(fit_values < FIT_THRESHOLD).sum()}개', fontsize=11)
low_by_class = collections.Counter(code for (fn, code), v in box_fit.items() if v < FIT_THRESHOLD)
n_by_class = collections.Counter(code for fn, code in box_fit)
order = sorted(low_by_class, key=lambda c: -low_by_class[c] / n_by_class[c])[:40]
axes[1].bar([f'K-{c:06d}' for c in order], [low_by_class[c] / n_by_class[c] for c in order], color='tab:orange')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)
axes[1].set_ylabel('box 맞음 0.75 미만 비율')
axes[1].set_title('클래스별 box 맞음 0.75 미만 비율 (높은 40개 클래스)', fontsize=11)
fig.suptitle(f'모든 AI Hub box의 box 맞음 (빨간 점선 = {FIT_THRESHOLD}, 이 아래는 "알약을 감싸지 않는 box"로 고친다), '
             f'믿을 수 없어 뺀 클래스: {", ".join(f"K-{c:06d}" for c in unreliable_classes) or "없음"}', fontsize=13)
save_fig(fig, '2-2_box_fit')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 2-2 표의 문제와 "알약을 감싸지 않는 box"마다 아래 규칙으로 고친다.
    - 문제가 있는 annotation은 모두 라벨 box에서 빼고, 남은 정상 box 밖에서 알약을 찾는다.
    - 찾은 알약과 문제 annotation을 크기와 색으로 짝짓는다. (위 검증의 box 둘 지우기와 같은 방법)
        - 크기·색 차이(`appearance_cost`)가 1.5를 넘는 짝은 만들지 않는다. 정상 라벨 box의 99.5%가 자기 클래스와 1.5 이하이다.
        - 사진에서 보이지 않는 알약의 클래스가 남은 다른 알약을 차지하지 않게 하려는 것이다. (예: 파란 알약 클래스가 옆의 주황 알약에 짝지어짐)
    - 원본 bbox가 있는 문제(이미지 밖, area 불일치, 숫자가 붙은 bbox, 알약을 감싸지 않는 box)는 원본 값을 최대한 살린다.
        - 원본 bbox와 찾은 알약 box의 IoU가 0.8 이상이면 bbox는 그대로 둔다. area 값만 틀린 것이다.
        - 아니면 오타를 고친 후보(숫자 하나를 빼거나, 넣거나, 바꾼 값, 쉼표가 빠져 붙은 숫자를 나눈 값)를 만든다.
            - 너비×높이가 원본 area와 같은 후보 중 찾은 알약 box와 IoU가 0.8 이상인 후보가 있으면, 그중 IoU가 가장 높은 값으로 고친다. ("숫자 하나 고침")
            - 그런 후보가 없으면 모든 후보 중 IoU가 0.85 이상인 값으로 고친다. 후보가 많아 우연히 맞는 값을 피하려고 기준을 높인다.
        - 맞는 후보가 없으면 찾은 알약 box로 바꾼다.
    - 다른 클래스와 box가 똑같은 두 annotation은 크기와 색으로 어느 클래스가 그 box의 알약인지 정한다.
        - 그 box의 알약으로 정해진 클래스는 원본 box를 그대로 쓰고, 다른 클래스는 찾은 알약 box를 쓴다.
    - 원본 bbox가 없는 문제(JSON 없음, bbox 빈 값, `-오류` 폴더, 읽을 수 없는 JSON)는 찾은 알약 box를 넣는다.
    - 짝지을 알약을 찾지 못하면, 쓸 수 있는 원본 bbox(이미지 안에 있는 값 4개)는 그대로 둔다.
        - 쓸 수 있는 원본 bbox가 없으면, 오타를 고친 후보 중 너비×높이가 원본 area와 같은 후보가 하나뿐일 때 그 값으로 고친다. ("숫자 하나 고침 (area로 확인)")
        - box가 똑같은 쌍에서 다른 클래스가 사진의 다른 알약으로 옮겨졌으면, 찾지 못한 쪽이 원본 box의 알약이므로 원본 box를 그대로 둔다.
        - 사진으로 확인하지 못한 box이므로, box의 긴 변이 그 클래스 라벨 box 긴 변의 중앙값과 25% 넘게 다르면 쓰지 않는다.
            - 알약을 돌려도 box의 긴 변은 거의 같아서, 같은 클래스의 정상 라벨 box 중 25% 넘게 다른 box는 0.05%이다.
        - 그래도 없으면 그 이미지를 학습에서 뺀다.
- 사진으로 다시 찾을 수 없는 경우는 찾은 알약으로 쓰지 않는다. 처음 만든 갤러리에서 이런 box가 알약이 아닌 자리에 그려졌다.
    - 사진의 두 변에 닿는 덩어리(조명 밖의 밝은 모서리)는 알약으로 보지 않는다.
    - 위 검증에서 IoU 중앙값이 0.85 미만인 클래스(예: 바닥과 색이 같은 파란 캡슐)는 짝짓기에서 빼고 "찾지 못함"과 같은 규칙으로 처리한다.

In [ ]:
def digit_edits(value):
    """숫자 하나에서 숫자 한 개를 빼거나, 넣거나, 바꾼 값을 모두 만든다."""
    text = str(int(value))
    out = set()
    for i in range(len(text) + 1):
        if i < len(text):
            out.add(text[:i] + text[i + 1:])
            out.update(text[:i] + d + text[i + 1:] for d in '0123456789')
        out.update(text[:i] + d + text[i:] for d in '0123456789')
    return {int(t) for t in out if t and not (len(t) > 1 and t[0] == '0')}


def raw_candidates(raw):
    """원본 bbox 값에서 오타를 고친 [x, y, w, h] 후보를 만든다. 이미지 안에 들어가는 후보만 남긴다.

    - 값이 4개인 정수: x, y, w, h 중 하나에서 숫자 한 개를 빼거나, 넣거나, 바꾼 값
    - 값이 3개 이하: 소수점이나 붙은 숫자를 둘로 나눠 값 4개를 만든다. (쉼표가 빠진 것으로 본다. 예: 128.816 → 128, 816)
    """
    tokens = []
    for v in raw:
        tokens += str(v).split('.') if isinstance(v, float) else [str(v)]
    bases = []
    if len(tokens) == 4:
        bases.append([int(t) for t in tokens])
    elif len(tokens) == 3:
        bases += [[int(s) for s in tokens[:i] + [t[:cut], t[cut:]] + tokens[i + 1:]]
                  for i, t in enumerate(tokens) for cut in range(1, len(t))]
    cands = {tuple(b) for b in bases}
    if len(raw) == 4 and all(isinstance(v, int) for v in raw):
        cands |= {tuple(raw[:i] + [v] + raw[i + 1:]) for i in range(4) for v in digit_edits(raw[i])}
    return [list(c) for c in cands if c[2] >= 30 and c[3] >= 30 and c[0] >= 0 and c[1] >= 0
            and c[0] + c[2] <= IMG_W and c[1] + c[3] <= IMG_H]


def choose_box(raw_box, raw_area, found):
    """원본 bbox를 살릴 수 있으면 (box, 처리)를, 없으면 (찾은 box, '사진에서 다시 그림')을 돌려준다."""
    if raw_box and len(raw_box) == 4 and all(isinstance(v, int) for v in raw_box) and raw_box in raw_candidates(raw_box) \
            and box_iou(raw_box, found) >= 0.8:
        return raw_box, '그대로 둠'
    cands = raw_candidates(raw_box) if raw_box else []
    by_area = [c for c in cands if raw_area and c[2] * c[3] == raw_area]
    for pool, threshold in [(by_area, 0.8), (cands, 0.85)]:
        if pool:
            best = max(pool, key=lambda c: box_iou(c, found))
            if box_iou(best, found) >= threshold:
                return best, '숫자 하나 고침'
    return found, '사진에서 다시 그림'


CORNER_GAP = 10  # 알약 범위가 사진의 두 변에서 이 픽셀 안이면 모서리 덩어리로 본다
MAX_COST = 1.5  # 크기·색 차이가 이보다 큰 알약과는 짝짓지 않는다 (정상 라벨 box의 99.5%가 이 안)
LONG_TOLERANCE = 0.25  # 사진으로 확인하지 못한 box는 긴 변이 클래스 중앙값과 이 비율 안일 때만 쓴다
long_values = collections.defaultdict(list)  # K-코드 → 문제가 없는 이미지의 라벨 box 긴 변
for fn in clean_pool:
    for ann in aihub_kaggle[fn]['annotations']:
        long_values[ann['category_id']].append(max(ann['bbox'][2], ann['bbox'][3]))
class_long = {code: float(np.median(v)) for code, v in long_values.items()}  # K-코드 → 긴 변의 중앙값
unfindable_classes = {c for c, v in validation_by_class.items() if np.median(v) < 0.85}  # 위 검증에서 사진으로 다시 찾지 못하는 클래스


def fix_candidates(rgb, known_boxes):
    """정상 box 밖에서 알약 범위를 찾고, 사진의 두 변에 닿는 모서리 덩어리(조명 밖의 밝은 모서리)는 뺀다."""
    out = []
    for extent in find_pills(rgb, known_boxes):
        x0, y0, x1, y1 = extent
        touches_x = x0 <= CORNER_GAP or x1 >= IMG_W - CORNER_GAP
        touches_y = y0 <= CORNER_GAP or y1 >= IMG_H - CORNER_GAP
        if not (touches_x and touches_y):
            out.append(extent)
    return out


def usable_raw(raw_box):
    """원본 bbox를 고치지 않고 쓸 수 있는지(이미지 안에 있는 정수 4개) 확인한다."""
    return bool(raw_box) and len(raw_box) == 4 and all(isinstance(v, int) for v in raw_box) and raw_box in raw_candidates(raw_box)


def plausible(box, code):
    """box의 긴 변이 그 클래스 라벨 box 긴 변의 중앙값과 LONG_TOLERANCE 안으로 비슷한지 본다. 중앙값이 없는 클래스는 통과시킨다."""
    return code not in class_long or abs(np.log(max(box[2], box[3]) / class_long[code])) <= np.log(1 + LONG_TOLERANCE)


def solve_image(fn):
    """이미지 하나의 문제 annotation을 고친다. 결과는 문제마다 dict 하나이다."""
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    items = problems[fn]
    bad = {code for code, *_ in items}
    good_boxes = [a['bbox'] for a in aihub_kaggle.get(fn, {'annotations': []})['annotations'] if a['category_id'] not in bad]
    extents = fix_candidates(rgb, good_boxes)
    labs = [extent_color(rgb, e) for e in extents]
    searchable = [i for i, (code, *_) in enumerate(items) if code not in unfindable_classes]  # 사진으로 찾을 수 있는 클래스만 짝짓는다
    appear = np.zeros((len(searchable), len(extents)))  # 크기·색 차이
    cost = np.zeros((len(searchable), len(extents)))  # 짝짓기 비용 = 크기·색 차이 - 원본 bbox 후보와의 IoU
    for row, i in enumerate(searchable):
        code, kind, raw_box, raw_area = items[i]
        ref = class_refs.get(code, no_ref)
        for j, extent in enumerate(extents):
            appear[row, j] = appearance_cost(extent, labs[j], ref) if 'long' in ref else 0
            cost[row, j] = appear[row, j]
            if raw_box and '똑같음' not in kind:  # 원본 bbox가 가리키는 알약이면 짝지을 때 우선한다
                cost[row, j] -= max([box_iou(c, with_margin(extent, ref['margin'])) for c in raw_candidates(raw_box)], default=0)
    cost[appear > MAX_COST] = 1e6  # 크기·색이 너무 다른 짝은 만들지 않는다
    results = []
    pairs = zip(*scipy.optimize.linear_sum_assignment(cost)) if extents and searchable else []
    assigned = {searchable[row]: j for row, j in pairs if appear[row, j] <= MAX_COST}
    for i, (code, kind, raw_box, raw_area) in enumerate(items):
        result = {'file_name': fn, 'category_id': code, 'problem': kind, 'raw_box': raw_box, 'raw_area': raw_area}
        if i not in assigned:
            area_matched = [c for c in raw_candidates(raw_box) if c[2] * c[3] == raw_area and plausible(c, code)] if raw_box and raw_area else []
            if usable_raw(raw_box) and plausible(raw_box, code) and '똑같음' not in kind:
                box, action = raw_box, '그대로 둠 (알약을 찾지 못함)'
            elif len(area_matched) == 1 and '똑같음' not in kind:
                box, action = area_matched[0], '숫자 하나 고침 (area로 확인)'
            else:
                box, action = None, '찾지 못함'
            results.append({**result, 'box': box, 'cost': None, 'action': action})
            continue
        found = with_margin(extents[assigned[i]], class_refs.get(code, no_ref)['margin'])
        if '똑같음' in kind:
            box, action = (raw_box, '그대로 둠') if box_iou(raw_box, found) >= 0.5 else (found, '사진에서 다시 그림')
        else:
            box, action = choose_box(raw_box, raw_area, found)
        results.append({**result, 'box': box, 'found': found, 'action': action, 'cost': float(cost[searchable.index(i), assigned[i]])})
    # box가 똑같은 쌍에서 다른 클래스가 다른 알약으로 옮겨졌으면, 찾지 못한 쪽이 원본 box의 알약이다
    moved = {tuple(r['raw_box']) for r in results if '똑같음' in r['problem'] and r['action'] == '사진에서 다시 그림'}
    for r in results:
        if ('똑같음' in r['problem'] and r['action'] == '찾지 못함' and tuple(r['raw_box']) in moved and usable_raw(r['raw_box'])
                and plausible(r['raw_box'], r['category_id'])):
            r['box'], r['action'] = r['raw_box'], '그대로 둠 (알약을 찾지 못함)'
    return results


fix_cache = CACHE_DIR / '2-2_fixes.json'
problem_key = json.dumps(sorted((fn, items) for fn, items in problems.items()))
if fix_cache.exists() and json.loads(fix_cache.read_text())['key'] == problem_key:
    box_fixes = json.loads(fix_cache.read_text())['fixes']
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        box_fixes = [r for results in pool.map(solve_image, sorted(problems)) for r in results]
    fix_cache.write_text(json.dumps({'key': problem_key, 'fixes': box_fixes}))

kinds = [k for k, _ in kind_count.most_common()]
actions = ['그대로 둠', '그대로 둠 (알약을 찾지 못함)', '숫자 하나 고침', '숫자 하나 고침 (area로 확인)', '사진에서 다시 그림', '찾지 못함']
table = collections.Counter((f['problem'], f['action']) for f in box_fixes)
rows = [[kind, *[table[(kind, a)] or '-' for a in actions]] for kind in kinds]
rows.append(['**합계**', *[f'**{sum(table[(k, a)] for k in kinds)}**' for a in actions]])
show_table(['문제', *actions], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 2-2에서 문제가 있는 이미지를 모두 사진 한 장씩 그려, 문제가 실제로 있는지와 고친 box가 문제를 해결하는지 눈으로 확인한다.
    - 빨간 점선: 원본 bbox (값이 4개일 때만 그린다. 이미지 밖으로 나간 부분은 잘려 보인다)
        - 고친 box와 다르면 오른쪽 아래에 "원본 K-코드"를 적는다.
    - 주황 점선: OpenCV로 사진에서 찾은 알약 box (고친 box와 다를 때만 그린다)
    - 파란 실선: 고친 box. 안쪽 위에 K-코드와 각인을 적는다.
    - 연두 실선: 같은 이미지에서 문제가 없는 라벨 box. 안쪽 아래에 K-코드를 적는다.
    - 제목: 번호, 파일명, 문제 annotation마다 "K-코드 문제: 원본 bbox → 처리: 고친 box". 고치지 못해 이미지를 빼는 경우 제목을 빨간색으로 쓴다.
- 이미지는 문제 순서(라벨 누락 → 이미지 밖 → 다른 클래스와 box가 똑같음 → area 불일치 → 알약을 감싸지 않는 box)로 놓는다.
    - 문제가 여러 개인 이미지는 앞 순서의 문제로 묶는다.
- 시트는 `runs/eda_raewoo/figures_01/2-2_problem_gallery/`에 8장씩 저장하고, 아래에는 첫 시트만 보인다.
    - 번호 → 파일명, K-코드, 문제, 처리, 원본 bbox, 고친 box의 색인은 같은 폴더의 `index.csv`이다.
    - 이미지 수가 바뀌면 시트 수도 바뀌므로, 저장하기 전에 옛 시트를 지운다.
- 원본 bbox가 정말 틀렸는지, 파란 box가 그 클래스의 알약 하나를 빈틈없이 감싸는지, 사진의 알약이 모두 연두나 파란 box에 들어갔는지 본다.

In [ ]:
PROBLEM_GALLERY_DIR = FIG_DIR / '2-2_problem_gallery'
PROBLEM_ROWS, PROBLEM_COLS = 2, 4  # 시트 한 장에 이미지 8장
PROBLEM_ORDER = ['라벨 누락', '이미지 밖', '다른 클래스와 box가 똑같음', 'area 불일치', '알약을 감싸지 않는 box']  # 시트에 놓는 순서


def problem_rank(problem):
    """문제 이름에 들어 있는 PROBLEM_ORDER 항목 중 가장 앞 순서를 돌려준다."""
    return min(i for i, name in enumerate(PROBLEM_ORDER) if name in problem)


def drawable(box):
    """사진에 그릴 수 있는 xywh box(숫자 4개, 너비와 높이가 양수)인지 확인한다."""
    return bool(box) and len(box) == 4 and all(isinstance(v, (int, float)) for v in box) and box[2] > 0 and box[3] > 0


def problem_panel(ax, n, fn, fixes):
    """문제 이미지 한 장에 문제없는 라벨 box, 원본 bbox, 사진에서 찾은 box, 고친 box를 그린다."""
    # 사진은 절반 크기로 줄여 그리고, extent로 원본 좌표계에 맞춰 box는 원본 좌표 그대로 그린다
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn).convert('RGB').reduce(2), extent=(0, IMG_W, IMG_H, 0))
    bad = {fix['category_id'] for fix in fixes}
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if ann['category_id'] not in bad:
            x, y, w, h = ann['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=1.5))
            ax.text(x + 4, y + h - 4, f"K-{ann['category_id']:06d}", fontsize=6, va='bottom',
                    bbox=dict(facecolor='lime', alpha=0.8, pad=1, linewidth=0), clip_on=True)
    lines = []
    for fix in fixes:
        code = fix['category_id']
        if drawable(fix['raw_box']):
            x, y, w, h = fix['raw_box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=2, linestyle='--'))
            # 원본 bbox가 고친 box와 다르고 사진에 걸쳐 있으면, 사진 안에 보이는 오른쪽 아래 모서리에 K-코드를 적는다
            if fix['raw_box'] != fix['box'] and x < IMG_W and y < IMG_H and x + w > 0 and y + h > 0:
                ax.text(min(x + w, IMG_W) - 4, min(y + h, IMG_H) - 4, f'원본 K-{code:06d}', fontsize=6, ha='right', va='bottom',
                        color='white', bbox=dict(facecolor='red', alpha=0.8, pad=1, linewidth=0), clip_on=True)
        if fix.get('found') and fix['found'] != fix['box']:
            x, y, w, h = fix['found']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='orange', linewidth=2, linestyle=':'))
        if fix['box']:
            x, y, w, h = fix['box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='blue', linewidth=2.5))
            label = pill_label(aihub_example[code]) if code in aihub_example else f'K-{code:06d}'
            ax.text(max(x, 0) + 4, max(y, 0) + 4, label, fontsize=7, va='top', color='white',
                    bbox=dict(facecolor='blue', alpha=0.8, pad=1, linewidth=0))
        after = '이미지 제외' if fix['action'] == '찾지 못함' else fix['box']
        lines.append(f"K-{code:06d} {fix['problem']}: {fix['raw_box']}\n    → {fix['action']}: {after}")
    ax.set_xlim(0, IMG_W)
    ax.set_ylim(IMG_H, 0)
    excluded = any(fix['action'] == '찾지 못함' for fix in fixes)
    ax.set_title(f'#{n} {fn}\n' + '\n'.join(lines), fontsize=7, loc='left', color='red' if excluded else 'black')
    ax.set_xticks([])
    ax.set_yticks([])


fixes_by_file = collections.defaultdict(list)  # 이미지 → 그 이미지의 문제 annotation 처리 결과
for fix in box_fixes:
    fixes_by_file[fix['file_name']].append(fix)
problem_files = sorted(fixes_by_file, key=lambda fn: (min(problem_rank(f['problem']) for f in fixes_by_file[fn]), fn))
per_sheet = PROBLEM_ROWS * PROBLEM_COLS
n_problem_sheets = int(np.ceil(len(problem_files) / per_sheet))
PROBLEM_GALLERY_DIR.mkdir(exist_ok=True)
for old_sheet in PROBLEM_GALLERY_DIR.glob('p*.jpg'):  # 이미지 수가 줄어 남는 옛 시트가 없게 지운다
    old_sheet.unlink()
problem_index = []
for page in range(n_problem_sheets):
    chunk = problem_files[page * per_sheet:(page + 1) * per_sheet]
    sheet_name = f'p{page + 1:03d}.jpg'
    fig, axes = plt.subplots(PROBLEM_ROWS, PROBLEM_COLS, figsize=(20, 15.5), layout='constrained')
    for k, (ax, fn) in enumerate(zip(axes.flat, chunk)):
        n = page * per_sheet + k + 1
        problem_panel(ax, n, fn, fixes_by_file[fn])
        problem_index += [{'sheet': sheet_name, 'number': n, 'file_name': fn, 'category_id': fix['category_id'],
                           'problem': fix['problem'], 'action': fix['action'], 'bbox_before': json.dumps(fix['raw_box']),
                           'bbox_after': json.dumps(fix['box']), 'found_box': json.dumps(fix.get('found'))}
                          for fix in fixes_by_file[fn]]
    for ax in axes.flat[len(chunk):]:
        ax.axis('off')
    groups = ', '.join(name for i, name in enumerate(PROBLEM_ORDER)
                       if any(min(problem_rank(f['problem']) for f in fixes_by_file[fn]) == i for fn in chunk))
    fig.suptitle(f'2-2 문제 이미지 {page * per_sheet + 1}~{page * per_sheet + len(chunk)}번째 / {len(problem_files)}장 '
                 f'(시트 {page + 1}/{n_problem_sheets}, {groups})\n'
                 '빨간 점선 = 원본 bbox, 주황 점선 = 사진에서 찾은 box, 파란 실선 = 고친 box, 연두 = 문제없는 라벨 box, '
                 '빨간 제목 = 이미지 제외', fontsize=14)
    fig.savefig(PROBLEM_GALLERY_DIR / sheet_name, dpi=90)
    if page:
        plt.close(fig)
with open(PROBLEM_GALLERY_DIR / 'index.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['sheet', 'number', 'file_name', 'category_id', 'problem', 'action',
                                           'bbox_before', 'bbox_after', 'found_box'])
    writer.writeheader()
    writer.writerows(problem_index)
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- `-오류` 폴더의 JSON이 왜 오류인지 확인한다.
    - 표: `-오류` 폴더 JSON마다 폴더 이름, drug_N, 제품명, bbox를 본다.
    - 그림: 그 이미지와 같은 조합의 각도별 사진 3장에 라벨 box(연두)와 2-2에서 찾은 box(파랑)를 그린다.
- 그 알약이 사진에 있는지, 다른 알약이 놓였는지, 가려졌는지 본다.

In [ ]:
error_jsons = sorted(p for p in raw_json_paths if unicodedata.normalize('NFC', p.parent.name).endswith('-오류'))
rows = []
for path in error_jsons:
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        rows.append([f'`{path.name}`', unicodedata.normalize('NFC', path.parent.name), '(읽을 수 없음)', '', ''])
        continue
    image = data['images'][0]
    rows.append([f'`{path.name}`', unicodedata.normalize('NFC', path.parent.name), image['drug_N'], image['dl_name'],
                 [a.get('bbox') for a in data['annotations']]])
show_table(['JSON', '폴더', 'drug_N', '제품명', 'bbox'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

In [ ]:
error_files = sorted({path.name.replace('.json', '.png') for path in error_jsons})
fig, axes = plt.subplots(int(np.ceil(len(error_files) / 3)), 3, figsize=(20, 9 * np.ceil(len(error_files) / 3)), layout='constrained', squeeze=False)
for ax, fn in zip(axes.flat, error_files):  # fixes_by_file은 위 문제 갤러리 셀에서 만든다
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
        ax.text(x, y - 8, pill_label(ann), fontsize=8, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
    for fix in fixes_by_file[fn]:
        if fix['box']:
            x, y, w, h = fix['box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='blue', linewidth=2.5))
            ax.text(x, y + h + 8, f"찾은 box: K-{fix['category_id']:06d}", fontsize=8, va='top', color='white',
                    bbox=dict(facecolor='blue', alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f'{fn}\nbbox 없음: {missing_text(fn)}', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.flat[len(error_files):]:
    ax.axis('off')
fig.suptitle('-오류 폴더 JSON이 있는 이미지 (연두 = 라벨 box, 파랑 = 2-2에서 사진으로 찾은 box)', fontsize=14)
save_fig(fig, '2-2_error_folder')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 2-3. 크기·색 대조와 클래스별 crop 갤러리
- 2-2의 수정을 적용한 라벨(`fixed_records`)에서, box 안의 알약이 그 클래스가 맞는지 두 가지로 본다.
    - 크기·색 대조: 이미지마다 box 안의 알약 범위를 재서(2-2의 `refine_extent`), 크기와 색이 2-2의 클래스 기준값과 가장 잘 맞도록 클래스를 다시 배정한다.
        - 다시 배정한 클래스가 라벨과 다르면, 라벨이 서로 바뀌었거나 box가 다른 자리에 있는 이미지 후보이다.
        - 배정 방법은 2-2의 box 둘 지우기 검증과 같다. (클래스 배정 정확도는 2-2 그림 제목)
    - crop 갤러리: 모든 box를 클래스별로 잘라 늘어놓고 눈으로 본다. (아래)
- 2-2에서 짝지을 알약을 찾지 못한 이미지는 뺀다. 알약 범위와 색은 2-2의 box 맞음 검사에서 잰 값을 쓴다.

In [ ]:
def drawn_annotation(fn, code, box):
    """원본 annotation이 없던 알약의 annotation을 만든다. 알약 메타데이터는 같은 클래스의 다른 annotation에서,
    이미지 필드(파일명, 촬영 각도 등)는 그 이미지의 것으로 채운다. (00 2-5와 같은 방법)"""
    example = aihub_example[code]
    image_fields = (aihub_kaggle[fn]['image'] if fn in aihub_kaggle
                    else {'file_name': fn, 'imgfile': fn, 'camera_la': int(fn.split('_')[5]), 'id': None})
    image = {**example['image'], **{k: image_fields[k] for k in IMAGE_FIELDS if k in image_fields}}
    return {**example, 'id': None, 'image_id': image.get('id'), 'json_path': None, 'bbox': list(box), 'image': image}


fixed_records = {}
fix_lookup = collections.defaultdict(dict)
for fix in box_fixes:
    fix_lookup[fix['file_name']][fix['category_id']] = fix
unfound_files = {f['file_name'] for f in box_fixes if f['action'] == '찾지 못함'}
for fn in sorted(aihub_records):
    if fn in unfound_files or fn not in set(aihub_files):
        continue
    anns = []
    by_code = {a['category_id']: a for a in aihub_kaggle.get(fn, {'annotations': []})['annotations']}
    for code in pill_codes(fn):
        fix = fix_lookup[fn].get(code)
        if fix:
            anns.append({**by_code[code], 'bbox': list(fix['box'])} if code in by_code else drawn_annotation(fn, code, fix['box']))
        elif code in by_code:
            anns.append(by_code[code])
    for ann in anns:
        ann['area'] = ann['bbox'][2] * ann['bbox'][3]
    fixed_records[fn] = {'image': anns[0]['image'] if anns else {}, 'annotations': anns}


fixed_measures = measure_all(fixed_records)  # 2-2에서 잰 값을 다시 쓰고, 고친 box만 새로 잰다


def appearance_check(fn):
    """크기·색으로 다시 배정한 클래스 순서와 (라벨 그대로의 비용, 가장 맞는 배정의 비용)을 구한다. 대조할 수 없으면 None이다."""
    anns = fixed_records[fn]['annotations']
    codes = [a['category_id'] for a in anns]
    measures = [fixed_measures[(fn, tuple(a['bbox']))] for a in anns]
    if len(anns) < 2 or any(c not in class_refs for c in codes) or any(m[0] is None for m in measures):
        return None
    cost = np.array([[appearance_cost(extent, lab, class_refs[c]) for c in codes] for extent, lab in measures])
    _, order = scipy.optimize.linear_sum_assignment(cost)
    return {'order': [int(j) for j in order], 'label_cost': float(np.trace(cost)), 'best_cost': float(cost[np.arange(len(codes)), order].sum())}


appearance = {fn: appearance_check(fn) for fn in sorted(fixed_records)}
checked = {fn: r for fn, r in appearance.items() if r}
mismatched = sorted((fn for fn, r in checked.items() if r['order'] != list(range(len(r['order'])))),
                    key=lambda fn: -(checked[fn]['label_cost'] - checked[fn]['best_cost']))
show_table(['항목', '이미지 수'], [
    ['크기·색 대조를 한 이미지', f'{len(checked):,}장'],
    ['대조하지 못한 이미지 (알약 1개 이하, 기준값이 없는 클래스, 알약 범위를 못 찾은 box)', f'{len(appearance) - len(checked):,}장'],
    ['다시 배정한 클래스가 라벨과 다른 이미지', f'{len(mismatched):,}장 ({len(mismatched) / len(checked):.2%})'],
])

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 다시 배정한 클래스가 라벨과 다른 이미지를 모두 그려 시트로 저장한다. 라벨 비용과 가장 맞는 배정의 비용 차이가 큰 순서이다.
    - 연두 box가 라벨이다. box 위 글자는 라벨의 K-코드(각인)와, 크기·색으로 다시 배정한 K-코드이다. 둘이 다르면 빨간 글자이다.
    - 시트는 `runs/eda_raewoo/figures_01/2-3_appearance_sheets/`에 8장씩 저장하고, 아래에는 첫 시트만 보인다.
- 라벨이 서로 바뀐 것인지, box가 다른 자리에 있는 것인지, 비슷한 알약이라 다시 배정이 틀린 것인지 본다.

In [ ]:
APPEAR_DIR = FIG_DIR / '2-3_appearance_sheets'
APPEAR_DIR.mkdir(exist_ok=True)
for old in APPEAR_DIR.glob('*.jpg'):
    old.unlink()
for start in range(0, len(mismatched), 8):
    chunk = mismatched[start:start + 8]
    fig, axes = plt.subplots(2, 4, figsize=(24, 17), layout='constrained')
    for ax, fn in zip(axes.flat, chunk):
        ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
        anns = fixed_records[fn]['annotations']
        order = checked[fn]['order']
        for i, ann in enumerate(anns):
            x, y, w, h = ann['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
            changed = order[i] != i
            ax.text(x, y - 8, f"{pill_label(ann)}\n→ 크기·색: K-{anns[order[i]]['category_id']:06d}", fontsize=8, va='bottom',
                    color='red' if changed else 'black', bbox=dict(facecolor='white', alpha=0.85, pad=1.5, linewidth=0))
        n = mismatched.index(fn) + 1
        ax.set_title(f"#{n} {fn}\n비용: 라벨 {checked[fn]['label_cost']:.2f} → 다시 배정 {checked[fn]['best_cost']:.2f}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes.flat[len(chunk):]:
        ax.axis('off')
    fig.suptitle(f'크기·색으로 다시 배정한 클래스가 라벨과 다른 이미지 {start + 1}~{start + len(chunk)} / {len(mismatched)}장 '
                 '(연두 = 라벨 box, 빨간 글자 = 라벨과 다르게 배정된 box)', fontsize=14)
    fig.savefig(APPEAR_DIR / f'2-3_appearance_p{start // 8 + 1:02d}.jpg', dpi=70)
    if start:
        plt.close(fig)
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- crop 갤러리: 2-2의 수정을 적용한 라벨(`fixed_records`)로, 모든 bbox를 클래스별로 잘라 시트에 늘어놓는다.
    - crop은 box 중심에서 긴 변보다 10% 큰 정사각형을 잘라 96×96으로 줄인 것이다. (00 2-3과 같은 `crop_square`) 노란 선이 라벨 box이다.
    - 클래스마다 box 넓이 순서로 놓는다. 크기가 튀는 box(헐거운 box, 옆 알약까지 감싼 box)는 시트의 처음이나 끝에 모인다.
    - 대회 train에도 있는 클래스는 첫 시트 첫 줄(빨간 테두리)에 대회 train crop을 놓는다. drug_N으로 매긴 클래스가 대회의 같은 알약인지 대조한다.
- 시트 한 장은 10×10 = 100칸이다. 시트는 `runs/eda_raewoo/figures_01/2-3_gallery/`에 `K-<코드>_p<쪽>.jpg`로 저장한다.
    - 칸마다 클래스 안 번호를 적는다. 번호로 이미지와 box를 찾는 색인은 같은 폴더의 `index.csv`이다.
- 1·2차 확인은 Claude Code가 시트를 모두 보고 하고, 애매한 칸은 사람이 확인한다. (00 5-2와 같은 분담)

In [ ]:
GALLERY_DIR = FIG_DIR / '2-3_gallery'
GALLERY_DIR.mkdir(exist_ok=True)
THUMB, COLS, ROWS = 96, 10, 10
gallery_items = collections.defaultdict(list)  # K-코드 → [(이미지, bbox)] box 넓이 순서
for fn, record in fixed_records.items():
    for ann in record['annotations']:
        gallery_items[ann['category_id']].append((fn, ann['bbox']))
for code in gallery_items:
    gallery_items[code].sort(key=lambda item: (item[1][2] * item[1][3], item[0]))
gallery_key = hashlib.md5(json.dumps(sorted(gallery_items.items())).encode()).hexdigest()
index_path = GALLERY_DIR / 'index.csv'
font_path = font_manager.findfont(plt.rcParams['font.family'][0])


def thumb_bytes(job):
    """이미지 하나에서 crop을 모두 잘라 JPEG 바이트로 돌려준다. (메모리를 아끼려고 압축해 둔다)"""
    folder, fn, boxes = job
    image = Image.open(folder / fn).convert('RGB')
    out = {}
    for box in boxes:
        x, y, w, h = box
        side = max(w, h) * 1.1  # crop_square와 같은 정사각형 (긴 변 + 10%)
        scale = THUMB / side
        thumb = crop_square(image, box, size=THUMB)
        left, top = THUMB / 2 - w * scale / 2, THUMB / 2 - h * scale / 2  # box는 정사각형 가운데에 온다
        ImageDraw.Draw(thumb).rectangle([left, top, left + w * scale, top + h * scale], outline=(255, 230, 0), width=1)
        buffer = io.BytesIO()
        thumb.save(buffer, format='JPEG', quality=90)
        out[(fn, tuple(box))] = buffer.getvalue()
    return out


def class_header(code, n_items, page, n_pages):
    info = (aihub_example.get(code) or class_example[code])['image']
    marks = '/'.join(m for m in (info.get('print_front'), info.get('print_back')) if m) or '각인 없음'
    return (f"K-{code:06d} {info['dl_name'][:28]} | 각인 {marks} | {info.get('color_class1')} {info.get('drug_shape')} | "
            f"AI Hub {n_items}개 | {page}/{n_pages}쪽")


cached = index_path.exists() and (GALLERY_DIR / 'key.txt').exists() and (GALLERY_DIR / 'key.txt').read_text() == gallery_key
if not cached:
    jobs = collections.defaultdict(list)
    for code, items in gallery_items.items():
        for fn, box in items:
            jobs[(AIHUB_IMG_DIR, fn)].append(box)
    for code in set(gallery_items) & set(train_classes):  # 대회 train crop (정제 라벨)
        for fn, record in clean_records.items():
            for ann in record['annotations']:
                if ann['category_id'] == code:
                    jobs[(TRAIN_IMG_DIR, fn)].append(ann['bbox'])
    thumbs_jpeg = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        for out in pool.map(thumb_bytes, [(folder, fn, boxes) for (folder, fn), boxes in jobs.items()]):
            thumbs_jpeg.update(out)
    font = ImageFont.truetype(font_path, 13)
    small_font = ImageFont.truetype(font_path, 11)
    index_rows = []
    for old in GALLERY_DIR.glob('K-*.jpg'):
        old.unlink()
    for code, items in sorted(gallery_items.items()):
        kaggle = [(fn, a['bbox']) for fn, r in sorted(clean_records.items()) for a in r['annotations'] if a['category_id'] == code][:COLS]
        first_capacity = COLS * (ROWS - 1) if kaggle else COLS * ROWS
        pages = [items[:first_capacity]] + [items[i:i + COLS * ROWS] for i in range(first_capacity, len(items), COLS * ROWS)]
        for p, page_items in enumerate(pages, start=1):
            sheet = Image.new('RGB', (COLS * THUMB, 24 + ROWS * (THUMB + 14)), 'white')
            draw = ImageDraw.Draw(sheet)
            draw.text((4, 4), class_header(code, len(items), p, len(pages)), fill='black', font=font)
            first_row = kaggle if p == 1 else []  # 첫 시트 첫 줄은 대회 train crop
            start_no = 0 if p == 1 else first_capacity + (p - 2) * COLS * ROWS
            for col, (fn, box) in enumerate(first_row):
                x0, y0 = col * THUMB, 24
                sheet.paste(Image.open(io.BytesIO(thumbs_jpeg[(fn, tuple(box))])), (x0, y0))
                draw.rectangle([x0, y0, x0 + THUMB - 1, y0 + THUMB - 1], outline='red', width=3)
                draw.text((x0 + 2, y0 + THUMB), '대회', fill='red', font=small_font)
            for k, (fn, box) in enumerate(page_items):
                row, col = divmod(k + (COLS if first_row else 0), COLS)
                x0, y0 = col * THUMB, 24 + row * (THUMB + 14)
                sheet.paste(Image.open(io.BytesIO(thumbs_jpeg[(fn, tuple(box))])), (x0, y0))
                number = start_no + k + 1
                draw.text((x0 + 2, y0 + THUMB), f'{number}', fill='black', font=small_font)
                index_rows.append({'sheet': f'K-{code:06d}_p{p:02d}.jpg', 'number': number, 'category_id': code,
                                   'file_name': fn, 'bbox': json.dumps(box)})
            sheet.save(GALLERY_DIR / f'K-{code:06d}_p{p:02d}.jpg', quality=85)
    with open(index_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['sheet', 'number', 'category_id', 'file_name', 'bbox'])
        writer.writeheader()
        writer.writerows(index_rows)
    (GALLERY_DIR / 'key.txt').write_text(gallery_key)
with open(index_path, encoding='utf-8') as f:
    gallery_index = list(csv.DictReader(f))
n_sheets = len(list(GALLERY_DIR.glob('K-*.jpg')))
show_table(['항목', '값'], [
    ['crop 수 (AI Hub, 2-2 수정 후)', f"{len(gallery_index):,}개 (이미지 {len(fixed_records):,}장)"],
    ['클래스 수', f'{len(gallery_items)}개 (대회 train crop 줄이 있는 클래스 {len(set(gallery_items) & set(train_classes))}개)'],
    ['시트 수', f'{n_sheets}장 (클래스당 {min(len(v) for v in gallery_items.values()) // 100 + 1}~{max(len(v) for v in gallery_items.values()) // 100 + 1}장)'],
    ['저장 위치', f'`{GALLERY_DIR.relative_to(PROJECT_ROOT)}`'],
])

- 시트 예시로 대회 train에도 있는 K-001900(보령부스파정 5mg)의 첫 시트를 보인다. (1-2의 예시 알약)

In [ ]:
example_sheet = Image.open(GALLERY_DIR / 'K-001900_p01.jpg')
fig, ax = plt.subplots(figsize=(14, 14 * example_sheet.height / example_sheet.width), layout='constrained')
ax.imshow(example_sheet)
ax.axis('off')
ax.set_title('crop 갤러리 시트 예시 (첫 줄 빨간 테두리 = 대회 train crop, 아래 = AI Hub crop을 box 넓이 순서로, 칸 아래 숫자 = 색인 번호)', fontsize=12)
save_fig(fig, '2-3_gallery_example')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 2-4. 라벨 정제 목록
- 2-1~2-3에서 찾은 라벨 오류를 annotation 단위로 모으고 처리 방법을 정한다.
    - 2-1, 2-2: 2-2의 규칙으로 고친 결과(`box_fixes`)이다.
    - 2-3: 크기·색 대조 시트와 갤러리 시트에서 눈으로 확인한 오류이다. (이미지, K-코드)마다 상수로 적는다.
        - 갤러리에서 찾은 칸은 시트 이름과 칸 번호로 `index.csv`에서 이미지를 찾는다.
        - 라벨이 서로 바뀐 알약은 맞는 K-코드로 바꾼다.
        - box가 틀린 알약(다른 자리, 옆 알약까지 감쌈, 알약을 다 감싸지 않음)은 2-2와 같은 방법으로 사진에서 다시 그린다.
        - 알약을 알아볼 수 없으면 그 이미지를 뺀다.
- 정제 목록은 00과 같은 열로 `notebooks/eda/01_raewoo_aihub_label_fixes.csv`에 저장한다. 바뀌는 annotation 하나가 한 줄이다.
    - `bbox_before`: 원본 JSON의 bbox 값 그대로이다. JSON이 없으면 비어 있고, 빈 값과 숫자가 붙은 값도 그대로 적는다.
    - `bbox_after`: 고친 `[x, y, w, h]` 픽셀이다. 이미지를 빼는 줄은 비어 있다.
    - `area_before`, `area_after`: 원본 area와 너비×높이로 다시 계산한 area이다.
    - `category_id_after`: 클래스를 바꾸는 줄에만 적는다.
    - `evidence`: 근거가 된 절 번호이다.

In [ ]:
LABEL_FINDINGS = {  # (이미지, K-코드) → (문제, 처리, 바꿀 K-코드). 처리는 '클래스 바꿈', 'box 다시 그림', '이미지 제외' 중 하나 (2-3 시트를 보고 적음)
}
gallery_fixes = []
for (fn, code), (problem, action, new_code) in LABEL_FINDINGS.items():
    ann = next(a for a in fixed_records[fn]['annotations'] if a['category_id'] == code)
    gallery_fixes.append({'file_name': fn, 'category_id': code, 'bbox': ann['bbox'], 'problem': problem, 'action': action,
                          'new_code': new_code})


def redraw_box(fn, code):
    """2-2의 방법으로 그 알약의 box를 사진에서 다시 찾는다. 찾지 못하면 None이다."""
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    others = [a['bbox'] for a in fixed_records[fn]['annotations'] if a['category_id'] != code]
    extents = fix_candidates(rgb, others)
    ref = class_refs.get(code)
    if not extents or ref is None:
        return None
    return with_margin(min(extents, key=lambda e: appearance_cost(e, extent_color(rgb, e), ref)), ref['margin'])


FIX_COLUMNS = ['file_name', 'category_id', 'problem', 'action', 'bbox_before', 'bbox_after',
               'area_before', 'area_after', 'category_id_after', 'evidence']
aihub_fix_rows = []
for fix in box_fixes:
    raw_box, box = fix['raw_box'], fix['box']
    if fix['action'] == '찾지 못함':
        action = '이미지 제외 (사진에서 알약을 찾지 못함)'
    elif fix['action'].startswith('그대로 둠'):
        if fix['raw_area'] in (None, box[2] * box[3]):
            continue  # bbox와 area 모두 바뀌지 않음 (box가 똑같은 두 클래스 중 그 box의 알약인 쪽)
        action = 'area를 너비×높이로 다시 계산'
    else:
        action = f"{fix['action']}: {box}"
    aihub_fix_rows.append({
        'file_name': fix['file_name'], 'category_id': fix['category_id'], 'problem': fix['problem'], 'action': action,
        'bbox_before': json.dumps(raw_box) if raw_box is not None else '', 'bbox_after': json.dumps(box) if box else '',
        'area_before': fix['raw_area'] if fix['raw_area'] is not None else '', 'area_after': box[2] * box[3] if box else '',
        'category_id_after': '', 'evidence': '2-2' if fix['problem'].startswith(('이미지 밖', 'area', '다른 클래스')) else '2-1, 2-2',
    })
for fix in gallery_fixes:
    box, new_code = fix['bbox'], fix['new_code']
    if fix['action'] == 'box 다시 그림':
        box = redraw_box(fix['file_name'], fix['category_id'])
    excluded = fix['action'] == '이미지 제외' or box is None
    aihub_fix_rows.append({
        'file_name': fix['file_name'], 'category_id': fix['category_id'], 'problem': fix['problem'],
        'action': '이미지 제외' if excluded else (f"box 다시 그림: {box}" if fix['action'] == 'box 다시 그림' else fix['action']),
        'bbox_before': json.dumps(fix['bbox']), 'bbox_after': '' if excluded else json.dumps(box),
        'area_before': fix['bbox'][2] * fix['bbox'][3], 'area_after': '' if excluded else box[2] * box[3],
        'category_id_after': new_code or '', 'evidence': '2-3',
    })
AIHUB_FIXES_CSV = EDA_DIR / '01_raewoo_aihub_label_fixes.csv'  # 전처리 코드가 그대로 적용하도록 git으로 공유한다
with open(AIHUB_FIXES_CSV, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=FIX_COLUMNS)
    writer.writeheader()
    writer.writerows(aihub_fix_rows)

action_kind = lambda a: a.split(':')[0]
summary = collections.Counter((row['evidence'], action_kind(row['action'])) for row in aihub_fix_rows)
rows = [[evidence, action, n, len({r['file_name'] for r in aihub_fix_rows if r['evidence'] == evidence and action_kind(r['action']) == action})]
        for (evidence, action), n in sorted(summary.items())]
rows.append(['**합계**', '', f'**{len(aihub_fix_rows)}줄**', f"**{len({r['file_name'] for r in aihub_fix_rows})}장**"])
show_table(['근거', '처리', 'annotation 수', '이미지 수'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 2-5. 라벨 정제 처리
- 2-4의 정제 목록을 바꾼 AI Hub 라벨(1-3)에 적용해 정제 라벨(`aihub_clean_records`)을 만든다. 3번부터는 이 라벨로 본다.
    - 목록의 `bbox_after`로 box를 바꾸고, `category_id_after`가 있으면 클래스와 알약 메타데이터를 그 클래스의 것으로 바꾼다.
    - 그려 넣은 box(원본 annotation이 없던 알약)의 알약 메타데이터는 같은 클래스의 다른 AI Hub annotation에서 가져온다.
    - 이미지를 빼는 줄이 있는 이미지는 모든 annotation을 뺀다.
    - 모든 area는 너비×높이로 다시 계산한다.
- 정제 전후의 이미지, 객체, 클래스 수를 비교한다.

In [ ]:
excluded_files = {row['file_name'] for row in aihub_fix_rows if row['action'].startswith('이미지 제외')}
fixes_by_image = collections.defaultdict(list)
for row in aihub_fix_rows:
    fixes_by_image[row['file_name']].append(row)
aihub_clean_records = {}
for fn in sorted(aihub_records):
    if fn in excluded_files or fn not in set(aihub_files):
        continue
    by_code = {a['category_id']: a for a in aihub_kaggle.get(fn, {'annotations': []})['annotations']}
    anns = []
    for code in pill_codes(fn):
        rows = [r for r in fixes_by_image[fn] if int(r['category_id']) == code and r['bbox_after']]
        if code in by_code:
            ann = dict(by_code[code])
        elif rows:
            ann = drawn_annotation(fn, code, json.loads(rows[0]['bbox_after']))
        else:
            continue  # 원본에도 없고 고친 box도 없는 알약 (이미지를 빼지 않았다면 생기지 않는다)
        for row in rows:  # 2-2의 수정 다음에 2-3의 수정을 적용한다 (목록 순서)
            ann['bbox'] = json.loads(row['bbox_after'])
            if row['category_id_after']:
                new_code = int(row['category_id_after'])
                ann['category_id'], ann['image'] = new_code, drawn_annotation(fn, new_code, ann['bbox'])['image']
        ann['area'] = ann['bbox'][2] * ann['bbox'][3]
        anns.append(ann)
    if anns:
        aihub_clean_records[fn] = {'image': anns[0]['image'], 'annotations': anns}


def label_summary(records):
    objects = [a for r in records.values() for a in r['annotations']]
    return [f'{len(records):,}장', f'{len(objects):,}개', f"{len({a['category_id'] for a in objects})}개",
            f"{sum(len(r['annotations']) < len(pill_codes(fn)) for fn, r in records.items()):,}장"]


show_table(['라벨', '이미지', '객체', '클래스', '파일명보다 알약이 적은 이미지'], [
    ['바꾼 AI Hub 라벨 (1-3)', *label_summary(aihub_kaggle)],
    ['정제 라벨 (2-5)', *label_summary(aihub_clean_records)],
])

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

## 3. bbox 분포
- 2-5의 정제 라벨(`aihub_clean_records`)과 정제한 대회 train(`clean_records`, 00 2-5)을 비교한다.
- 00 3절에서 정한 입력 크기(640으로 시작해 1280과 비교), 크기 조절·모자이크 증강(기본값), NMS IoU(0.7)가 AI Hub를 섞은 뒤에도 맞는지 본다.

### 3-1. 객체 수와 크기
- 왼쪽: 이미지당 객체 수의 비율을 비교한다.
- 가운데: box의 짧은 변 길이(원본 픽셀) 분포를 비교한다. 긴 알약은 놓인 방향에 따라 너비와 높이가 바뀌므로 짧은 변으로 본다.
- 오른쪽: 입력 640(긴 변 1280을 640으로 줄임, 0.5배)에서의 짧은 변 길이이다. 00 3-1에서 대회 train은 최소 62, 중앙값 107픽셀이었다.
- 세 그림 모두 묶음의 크기가 달라서 비율(밀도)로 그린다.

In [ ]:
def box_sides(records):
    """box마다 (짧은 변, 긴 변) 픽셀을 모은다."""
    return np.array([sorted(a['bbox'][2:]) for r in records.values() for a in r['annotations']], dtype=float)


compare_sets = {'대회 train (정제)': clean_records, 'AI Hub (정제)': aihub_clean_records}
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), layout='constrained')
for (name, records), color in zip(compare_sets.items(), ['tab:blue', 'tab:green']):
    counts = collections.Counter(len(r['annotations']) for r in records.values())
    ks = np.arange(1, 5)
    share = np.array([counts[k] for k in ks]) / len(records)
    offset = -0.2 if color == 'tab:blue' else 0.2
    axes[0].bar(ks + offset, share, width=0.4, color=color, label=f'{name} {len(records):,}장')
    for k, s in zip(ks, share):
        if s > 0:
            axes[0].text(k + offset, s + 0.01, f'{s:.1%}', ha='center', fontsize=9)
    sides = box_sides(records)
    axes[1].hist(sides[:, 0], bins=np.arange(0, 560, 10), density=True, histtype='step', linewidth=2, color=color,
                 label=f'{name} 중앙값 {np.median(sides[:, 0]):.0f}, 최소 {sides[:, 0].min():.0f}')
    axes[2].hist(sides[:, 0] * 0.5, bins=np.arange(0, 280, 5), density=True, histtype='step', linewidth=2, color=color,
                 label=f'{name} 중앙값 {np.median(sides[:, 0]) * 0.5:.0f}, 최소 {sides[:, 0].min() * 0.5:.0f}')
axes[0].set_xticks(ks)
axes[0].set_xlabel('이미지당 객체 수 (개)')
axes[0].set_ylabel('이미지 비율')
axes[0].set_title('이미지당 객체 수')
axes[1].set_xlabel('box 짧은 변 (원본 픽셀)')
axes[1].set_title('box 짧은 변 길이 (원본 976×1280)')
axes[2].set_xlabel('box 짧은 변 (입력 640 기준 픽셀)')
axes[2].set_title('입력 640에서의 box 짧은 변 길이')
for ax in axes[1:]:
    ax.set_ylabel('밀도 (넓이 합 = 1)')
for ax in axes:
    ax.legend(fontsize=9)
fig.suptitle('대회 train과 AI Hub의 객체 수와 box 크기 (묶음마다 비율로 그림)', fontsize=14)
save_fig(fig, '3-1_objects_and_sizes')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 같은 알약을 같은 각도에서 찍으면 알약이 사진에 같은 크기로 찍히므로, box 크기 차이는 라벨을 그린 습관(알약 둘레 여백)의 차이이다.
    - 대회 train과 AI Hub에 모두 있는 클래스에서, (클래스, 촬영 각도)마다 box의 긴 변과 짧은 변 중앙값을 비교한다.
    - 점이 대각선 위에 있으면 두 데이터의 box 크기가 같다. 대각선에서 벗어난 거리가 여백 차이이다.
- 00 8-2에서 box를 4픽셀 옮기면 mAP@[0.75:0.95]가 0.87로 떨어졌다. 여백 습관이 몇 픽셀만 달라도 test 점수가 깎인다.

In [ ]:
def side_medians(records):
    """(K-코드, 촬영 각도)마다 box의 긴 변, 짧은 변 중앙값과 box 수를 구한다."""
    sides = collections.defaultdict(list)
    for record in records.values():
        for ann in record['annotations']:
            sides[(ann['category_id'], ann['image']['camera_la'])].append(sorted(ann['bbox'][2:], reverse=True))
    return {key: (np.median(v, axis=0), len(v)) for key, v in sides.items()}


kaggle_sides, aihub_sides = side_medians(clean_records), side_medians(aihub_clean_records)
common_keys = sorted(set(kaggle_sides) & set(aihub_sides))
diffs = np.array([aihub_sides[k][0] - kaggle_sides[k][0] for k in common_keys])  # (긴 변 차이, 짧은 변 차이)
fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained')
for i, (ax, side) in enumerate(zip(axes, ['긴 변', '짧은 변'])):
    k_vals = np.array([kaggle_sides[k][0][i] for k in common_keys])
    a_vals = np.array([aihub_sides[k][0][i] for k in common_keys])
    ax.scatter(k_vals, a_vals, s=18, alpha=0.7)
    lo, hi = min(k_vals.min(), a_vals.min()) - 10, max(k_vals.max(), a_vals.max()) + 10
    ax.plot([lo, hi], [lo, hi], color='gray', linestyle='--', linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(f'대회 train box {side} 중앙값 (픽셀)')
    ax.set_ylabel(f'AI Hub box {side} 중앙값 (픽셀)')
    d = diffs[:, i]
    ax.set_title(f'{side}: AI Hub - 대회 차이 중앙값 {np.median(d):+.1f}픽셀, 절댓값 4픽셀 넘는 묶음 {(np.abs(d) > 4).sum()}/{len(d)}')
fig.suptitle(f'같은 클래스·같은 각도의 box 크기 비교 ({len(common_keys)}개 묶음, 점 = (클래스, 각도) 하나, 회색 점선 = 크기가 같음)', fontsize=14)
save_fig(fig, '3-1b_margin_compare')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 3-2. 위치와 겹침
- 왼쪽: box 중심 위치의 분포를 AI Hub 전체로 본다. (색이 진할수록 box 중심이 많음)
- 오른쪽: 이미지마다 box 쌍의 IoU 최댓값을 구해 분포를 비교한다.
    - 00 3-2에서 대회 train은 최댓값이 0.021이었다. 알약끼리 거의 겹치지 않아 NMS IoU를 기본값 0.7로 두기로 했다.

In [ ]:
centers = np.array([[a['bbox'][0] + a['bbox'][2] / 2, a['bbox'][1] + a['bbox'][3] / 2]
                    for r in aihub_clean_records.values() for a in r['annotations']])


def max_pair_iou(records):
    """이미지마다 box 쌍의 IoU 최댓값을 구한다. box가 하나면 0이다."""
    values = []
    for record in records.values():
        boxes = [a['bbox'] for a in record['annotations']]
        values.append(max([box_iou(a, b) for i, a in enumerate(boxes) for b in boxes[i + 1:]], default=0))
    return np.array(values)


fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained', gridspec_kw={'width_ratios': [0.8, 1.2]})
h = axes[0].hist2d(centers[:, 0], centers[:, 1], bins=[np.linspace(0, IMG_W, 25), np.linspace(0, IMG_H, 33)], cmap='Blues')
fig.colorbar(h[3], ax=axes[0], label='box 중심 수 (개)')
axes[0].set_xlim(0, IMG_W)
axes[0].set_ylim(IMG_H, 0)
axes[0].set_aspect('equal')
axes[0].set_xlabel('x (픽셀)')
axes[0].set_ylabel('y (픽셀)')
axes[0].set_title(f'AI Hub box 중심 위치 ({len(centers):,}개)')
bins = np.linspace(0, 1, 51)
pair_iou = {name: max_pair_iou(records) for name, records in compare_sets.items()}
for (name, values), color in zip(pair_iou.items(), ['tab:blue', 'tab:green']):
    axes[1].hist(values, bins=bins, histtype='step', linewidth=2, color=color,
                 label=f'{name} {len(values):,}장: 최댓값 {values.max():.3f}, 0.05 넘는 이미지 {(values > 0.05).sum()}장')
axes[1].set_yscale('log')
axes[1].set_xlabel('이미지 안 box 쌍의 IoU 최댓값')
axes[1].set_ylabel('이미지 수 (장, 로그 눈금)')
axes[1].set_title('이미지마다 가장 많이 겹친 box 쌍의 IoU')
axes[1].legend()
fig.suptitle('box 위치와 겹침', fontsize=14)
save_fig(fig, '3-2_position_overlap')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

## 4. 클래스 분포
- 2-5의 정제 라벨(`aihub_clean_records`)로 본다. 대회 train은 00 2-5의 정제 라벨(`clean_records`)이다.

### 4-1. 클래스별 객체 수와 등장 조합 수
- 위: 대회 train 클래스는 대회 train과 AI Hub의 객체 수를 나란히 놓고(로그 눈금), train에 없는 클래스는 AI Hub 객체 수만 본다.
- 아래: 같은 클래스들이 나오는 조합 수이다. 조합 하나는 같은 알약 배치를 각도만 바꿔 찍은 사진 3장이다. (00 4-3)
    - 객체 수가 많아도 조합 수가 적으면, 같은 배치의 사진만 반복해서 보는 셈이다.

In [ ]:
def class_stats(records):
    """클래스별 객체 수와 등장 조합 수를 센다."""
    objects, combos = collections.Counter(), collections.defaultdict(set)
    for fn, record in records.items():
        for ann in record['annotations']:
            objects[ann['category_id']] += 1
            combos[ann['category_id']].add(combo_name(fn))
    return objects, {c: len(s) for c, s in combos.items()}


aihub_object_count, aihub_combo_count = class_stats(aihub_clean_records)
_, train_combo_count = class_stats(clean_records)
aihub_classes = set(aihub_object_count)
new_classes = sorted(aihub_classes - set(train_classes), key=lambda c: (-aihub_object_count[c], c))
absent_train = sorted(set(train_classes) - aihub_classes)

fig, axes = plt.subplots(2, 2, figsize=(22, 10), layout='constrained', gridspec_kw={'width_ratios': [1, 1.1]})
order = sorted(train_classes, key=lambda c: (-clean_object_count[c], c))
x = np.arange(len(order))
for row, (train_values, aihub_values, unit) in enumerate([(clean_object_count, aihub_object_count, '객체 수'),
                                                           (train_combo_count, aihub_combo_count, '등장 조합 수')]):
    ax = axes[row, 0]
    ax.bar(x - 0.2, [train_values.get(c, 0) for c in order], width=0.4, label='대회 train (00 2-5 정제 라벨)')
    ax.bar(x + 0.2, [aihub_values.get(c, 0) for c in order], width=0.4, label='AI Hub (2-5 정제 라벨)')
    ax.set_yscale('log')
    ax.set_xticks(x, [f'K-{c:06d}' for c in order], rotation=90, fontsize=7)
    ax.set_ylabel(f'{unit} (개, 로그 눈금)')
    ax.set_title(f'대회 train 클래스 {len(train_classes)}개의 {unit}' +
                 (f'\nAI Hub에 없는 train 클래스: {", ".join(f"K-{c:06d}" for c in absent_train)} (막대 높이 0)' if row == 0 else ''))
    ax.legend()
    ax = axes[row, 1]
    values = [aihub_values[c] for c in new_classes]
    ax.bar([f'K-{c:06d}' for c in new_classes], values, color='tab:green')
    ax.tick_params(axis='x', rotation=90, labelsize=7)
    ax.set_ylabel(f'{unit} (개)')
    ax.set_title(f'train에 없는 클래스 {len(new_classes)}개의 AI Hub {unit} (최소 {min(values)}, 중앙값 {np.median(values):g})')
fig.suptitle(f'AI Hub 정제 라벨 {len(aihub_clean_records):,}장의 클래스별 객체 수와 등장 조합 수 (오른쪽은 객체 수 순서로 정렬)', fontsize=14)
save_fig(fig, '4-1_class_coverage')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 4-2. 조합당 이미지 수
- AI Hub 소개대로 조합마다 촬영 각도(camera_la) 70°, 75°, 90° 사진이 한 장씩 있는지 센다.
- 2-5에서 뺀 이미지 때문에 3장이 다 남지 않은 조합도 센다.

In [ ]:
def combo_angles(records):
    """조합마다 남은 사진의 촬영 각도를 모은다."""
    angles = collections.defaultdict(list)
    for fn, record in records.items():
        angles[combo_name(fn)].append(record['image']['camera_la'])
    return angles


rows = []
for name, records in [('AI Hub 라벨 zip 전체', {fn: {'image': {'camera_la': int(fn.split('_')[5])}} for fn in aihub_records}),
                      ('AI Hub 정제 라벨 (2-5)', aihub_clean_records), ('대회 train (참고)', clean_records)]:
    angles = combo_angles(records)
    per_combo = collections.Counter(len(v) for v in angles.values())
    full = sum(sorted(v) == [70, 75, 90] for v in angles.values())
    rows.append([name, f'{len(records):,}장', f'{len(angles):,}개', ', '.join(f'{k}장 {v:,}개' for k, v in sorted(per_combo.items())),
                 f'{full:,}개 ({full / len(angles):.1%})'])
show_table(['라벨', '이미지', '조합', '조합당 이미지 수', '70°·75°·90°가 모두 있는 조합'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 4-3. 같은 조합 사진 비교
- 무작위로 고른 조합 2개의 각도별 사진 3장을 나란히 놓고, 정제 라벨 box를 겹쳐 그린다. (seed 고정)
- 00 4-3에서 대회 train의 같은 조합 사진은 알약 배치가 같고, 70° 사진은 배치가 180° 돌아가 있었다. AI Hub도 같은지 본다.

In [ ]:
full_combos = sorted(c for c, v in combo_angles(aihub_clean_records).items() if sorted(v) == [70, 75, 90])
picked = [full_combos[i] for i in np.random.default_rng(SEED).choice(len(full_combos), size=2, replace=False)]
fig, axes = plt.subplots(2, 3, figsize=(20, 17), layout='constrained')
for row_axes, combo in zip(axes, picked):
    files = sorted((fn for fn in aihub_clean_records if combo_name(fn) == combo),
                   key=lambda fn: aihub_clean_records[fn]['image']['camera_la'])
    for ax, fn in zip(row_axes, files):
        ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
        for ann in aihub_clean_records[fn]['annotations']:
            x, y, w, h = ann['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
            ax.text(x, y - 8, pill_label(ann), fontsize=8, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
        ax.set_title(f"{aihub_clean_records[fn]['image']['camera_la']}° | {fn}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle('같은 조합의 각도별 사진 (행 = 조합, 열 = 촬영 각도 70°·75°·90°, 연두 box = 정제 라벨)', fontsize=14)
save_fig(fig, '4-3_same_combo')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 4-4. 대회 train과 합친 분포
- 정제한 대회 train에 AI Hub 정제 라벨을 모두 더했을 때 클래스 분포가 어떻게 바뀌는지 표로 본다.
- 클래스별 객체 수의 최대÷최소와 변동계수(표준편차÷평균, 0이면 모든 클래스의 객체 수가 같음)로 불균형 정도를 잰다.
- 목적: 클래스 가중치나 반복 샘플링 같은 불균형 대응이 필요한지 정한다.

In [ ]:
def class_list(objects, keep):
    """조건에 맞는 클래스를 객체 수 순서로 적는다. 6개 이상이면 개수만 적는다."""
    picked = sorted((n, c) for c, n in objects.items() if keep(n))
    if len(picked) > 5:
        return f'{len(picked)}개'
    return ', '.join(f'K-{c:06d} {n}개' for n, c in picked) or '없음'


def distribution_column(records):
    """클래스 분포 표의 한 열을 만든다."""
    objects, combos = class_stats(records)
    values = np.array(sorted(objects.values()))
    total = values.sum()
    top_code, top_n = objects.most_common(1)[0]
    return [
        f'{len(records):,}장', f'{total:,}개', f'{len(objects)}개',
        f'{values[0]} / {np.median(values):g} / {values[-1]}', f'{values[-1] / values[0]:.1f}배', f'{values.std() / values.mean():.2f}',
        f'K-{top_code:06d} {top_n}개 ({top_n / total:.1%})',
        ' / '.join(str(sum(lo <= n < hi for n in values)) for lo, hi in BINS),
        class_list(objects, lambda n: n < BINS[0][1]),
        f'{min(combos.values())} / {np.median(list(combos.values())):g}',
    ]


BINS = [(0, 100), (100, 200), (200, 400), (400, 800), (800, float('inf'))]  # 객체 수 구간
combined_records = {**clean_records, **aihub_clean_records}
assert not set(clean_records) & set(aihub_clean_records), '대회 train과 AI Hub에 같은 파일명이 있다 (1-6)'
row_names = ['이미지 수', '객체 수', '클래스 수', '클래스별 객체 수 (최소 / 중앙값 / 최대)', '최대 ÷ 최소', '변동계수 (표준편차 ÷ 평균)',
             '객체가 가장 많은 클래스 (비중)', '객체 수 구간별 클래스 수 (100 미만 / 100~199 / 200~399 / 400~799 / 800 이상)',
             '객체 100개 미만 클래스', '클래스별 등장 조합 수 (최소 / 중앙값)']
show_table(['항목', '대회 train만 (00 2-5)', f'대회 train + AI Hub 전부 (2-5)'],
           [[name, before, after] for name, before, after in zip(row_names, distribution_column(clean_records),
                                                                 distribution_column(combined_records))])

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 합친 분포를 클래스마다 막대 하나로 그린다.
    - 대회 train 클래스와 train에 없는 클래스를 색으로 나누고, 그룹 안에서는 객체 수 순서로 놓는다.
    - 대회 train 클래스 막대 아래쪽의 짙은 남색은 대회 원본 객체이다.

In [ ]:
def class_bar(ax, objects, title, line=None):
    """클래스별 객체 수를 그룹(대회 train 클래스, train에 없는 클래스)으로 나눠 막대로 그린다."""
    start, ticks = 0, []
    for name, classes, color in [('대회 train 클래스', set(train_classes), 'tab:blue'),
                                 ('train에 없는 클래스', set(objects) - set(train_classes), 'tab:green')]:
        order = sorted(classes, key=lambda c: (-objects[c], c))
        xs = np.arange(start, start + len(order))
        before = np.array([clean_object_count[c] for c in order])
        after = np.array([objects[c] for c in order])
        if before.any():
            ax.bar(xs, before, color='#08306b', label='대회 원본 객체 (00 2-5 정제 라벨)')
        ax.bar(xs, after - before, bottom=before, color=color,
               label=f'{name} {len(order)}개 (객체 {after.min()}~{after.max()}개, 중앙값 {np.median(after):g}개)')
        if start:
            ax.axvline(start - 0.5, color='gray', linewidth=0.8, linestyle=':')
        ticks += order
        start += len(order)
    if line:
        ax.axhline(line, color='black', linewidth=1, linestyle='--', label=f'목표 객체 수 {line}개')
    ax.set_xticks(np.arange(len(ticks)), [f'K-{c:06d}' for c in ticks], rotation=90, fontsize=6)
    ax.set_xlim(-1, len(ticks))
    ax.set_ylabel('객체 수 (개)')
    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=9)


combined_objects, _ = class_stats(combined_records)
fig, ax = plt.subplots(figsize=(24, 7), layout='constrained')
class_bar(ax, combined_objects, f'대회 train {len(clean_records)}장 + AI Hub 전부 {len(aihub_clean_records):,}장의 클래스 {len(combined_objects)}개 객체 수 '
          f'(최대÷최소 {max(combined_objects.values()) / min(combined_objects.values()):.1f}배)')
save_fig(fig, '4-4_combined_classes')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 4-5. 가장 균일하게 고르기
- 클래스마다 목표 객체 수를 정하고, 목표에 못 미친 클래스를 채우는 AI Hub 이미지만 고른다.
    - 사진에는 알약 3~4개가 함께 있어서, 이미지를 넣으면 그 안의 알약이 모두 함께 들어간다. 그래서 완전히 같은 수로 맞출 수는 없다.
    - 00 6-6의 방법(목표에 못 미친 클래스가 하나라도 있으면 넣음)은 함께 찍힌 흔한 클래스도 계속 늘린다.
    - 그래서 객체가 적은 클래스부터 하나씩, 그 클래스가 목표를 채울 때까지 그 클래스가 있는 이미지를 넣는다.
        - 한 클래스 안에서는 이미 목표를 채운 클래스가 적게 함께 찍힌 이미지부터 넣는다. 흔한 클래스가 덜 늘어난다.
        - 모든 클래스가 목표나, 목표보다 적으면 가진 객체 전부를 채운다.
- 목표 객체 수별로 고른 이미지 수와 불균형 정도(최대÷최소, 변동계수)를 AI Hub를 전부 쓸 때와 비교한다.
    - 00 6-6은 디스크 용량 때문에 목표를 골랐다. 지금은 이미지를 모두 받았으므로, 학습에 무엇을 넣을지의 문제이다.

In [ ]:
TARGETS = [100, 200, 300, 400]
image_classes = {fn: [a['category_id'] for a in r['annotations']] for fn, r in aihub_clean_records.items()}
all_classes = sorted(set(train_classes) | aihub_classes)


images_by_class = collections.defaultdict(list)  # K-코드 → 그 클래스가 있는 AI Hub 이미지
for fn, classes in sorted(image_classes.items()):
    for c in set(classes):
        images_by_class[c].append(fn)
total_objects = collections.Counter(clean_object_count) + collections.Counter(c for cs in image_classes.values() for c in cs)


def plan_uniform(target):
    """객체가 적은 클래스부터 목표 객체 수를 채우도록 AI Hub 이미지를 고른다."""
    count = collections.Counter(clean_object_count)  # 대회 train(00 2-5 정제 라벨)은 모두 넣고 시작한다
    chosen = set()
    for c in sorted(all_classes, key=lambda c: (total_objects[c], c)):
        # 이미 목표를 채운 클래스가 적게 함께 찍힌 이미지부터 넣는다
        pool = sorted((fn for fn in images_by_class[c] if fn not in chosen),
                      key=lambda fn: (sum(count[o] >= target for o in image_classes[fn]), fn))
        for fn in pool:
            if count[c] >= target:
                break
            chosen.add(fn)
            count.update(image_classes[fn])
    return sorted(chosen), count


def balance_row(name, objects, n_images):
    values = np.array([objects[c] for c in all_classes])
    return [name, f'{n_images:,}장', f'{values.sum():,}개', f'{values.min()} / {np.median(values):g} / {values.max()}',
            f'{values.max() / values.min():.1f}배', f'{values.std() / values.mean():.2f}']


uniform_plans = {t: plan_uniform(t) for t in TARGETS}
rows = [balance_row('AI Hub 전부', combined_objects, len(aihub_clean_records))]
for t, (chosen, count) in uniform_plans.items():
    short = sum(count[c] < t for c in all_classes)
    rows.append(balance_row(f'목표 {t}개 (못 미친 클래스 {short}개)', count, len(chosen)))
show_table(['AI Hub 이미지 선택', 'AI Hub 이미지 수', '객체 수 (대회 train 포함)', '클래스별 객체 수 (최소 / 중앙값 / 최대)', '최대 ÷ 최소', '변동계수'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

- 4-5 표에서 고른 목표 객체 수로 AI Hub 이미지 목록을 `data/external/aihub_targets.csv`에 저장하고, 전부 쓸 때와 클래스별 객체 수를 나란히 그린다.
    - 목록의 열은 00 6-6과 같다. (라벨 zip, 이미지 zip, filekey, 이미지 파일명)

In [ ]:
SELECTED_TARGET = 200  # 4-5 표를 보고 정한다
IMAGE_ZIP = {  # 라벨 zip → (이미지 zip, filekey)
    'TL_1_조합.zip': ('TS_1_조합.zip', 66154), 'TL_3_조합.zip': ('TS_3_조합.zip', 66156),
    'TL_4_조합.zip': ('TS_4_조합.zip', 66157), 'TL_5_조합.zip': ('TS_5_조합.zip', 66158),
    'TL_6_조합.zip': ('TS_6_조합.zip', 66159), 'TL_7_조합.zip': ('TS_7_조합.zip', 66160),
    'TL_8_조합.zip': ('TS_8_조합.zip', 66161), 'VL_1_조합.zip': ('VS_1_조합.zip', 66254),
}
selected, selected_count = uniform_plans[SELECTED_TARGET]
with open(EXTERNAL_DIR / 'aihub_targets.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['label_zip', 'image_zip', 'filekey', 'file_name'])
    for fn in sorted(selected, key=lambda n: (label_zip_of[n], n)):
        writer.writerow([label_zip_of[fn], *IMAGE_ZIP[label_zip_of[fn]], fn])

fig, axes = plt.subplots(2, 1, figsize=(24, 13), layout='constrained', sharex=False)
class_bar(axes[0], combined_objects, f'AI Hub 전부 {len(aihub_clean_records):,}장 (최대÷최소 {max(combined_objects.values()) / min(combined_objects.values()):.1f}배)')
class_bar(axes[1], selected_count, f'목표 {SELECTED_TARGET}개로 고른 AI Hub {len(selected):,}장 '
          f'(최대÷최소 {max(selected_count[c] for c in all_classes) / min(selected_count[c] for c in all_classes):.1f}배)', line=SELECTED_TARGET)
fig.suptitle('대회 train + AI Hub의 클래스별 객체 수: 전부 쓸 때(위)와 목표 객체 수로 고를 때(아래)', fontsize=14)
save_fig(fig, '4-5_uniform_selection')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

## 5. test 전용 클래스 채우기
- 00 5-3에서 test 사진을 직접 보고 train에 없는 알약 18종(각인 19종, 213과 30은 같은 알약의 앞뒤)을 찾았다. test 842장 중 335장(39.8%)에 나온다.
- 이 알약들이 AI Hub로 채워지는지 정제 라벨(2-5)로 확인한다.

### 5-1. test에서 읽은 각인 대조
- 00 5-3 기록표의 각인을 AI Hub 라벨의 각인 필드(print_front, print_back)와 마크 설명 필드(mark_code_front_anal, mark_code_back_anal)에서 찾는다.
    - 회사 로고처럼 그림으로 된 각인은 각인 필드에 "마크"라고만 적고, 로고의 글자는 마크 설명 필드에 적는다. (예: 각인 필드 "마크", 마크 설명 "Hanmi")
    - 00 6-5는 각인 필드만 찾아서 Hanmi를 찾지 못했다.
    - 띄어쓰기와 대소문자는 무시하고, 필드 값 전체가 같은 경우를 먼저 찾는다.
    - 같은 값이 없으면 필드 값에 각인이 들어 있는 경우(부분 일치)를 찾는다.
    - AI Hub는 분할선을 "분할선"이라는 글자로 적으므로, 기록표의 `|`를 바꿔서 찾는다. (K|50 → `K분할선50`)

In [ ]:
def normalize(text):
    """띄어쓰기와 대소문자를 무시하고 비교하도록 각인 글자를 정리한다."""
    return str(text or '').replace(' ', '').lower()


aihub_pill_info = {}  # K-코드 → 그 알약 JSON의 이미지 메타데이터
for record in aihub_clean_records.values():
    for ann in record['annotations']:
        aihub_pill_info.setdefault(ann['category_id'], ann['image'])
filled = sum(bool(im['print_front'] or im['print_back']) for im in aihub_pill_info.values())
IMPRINT_KEYS = ['print_front', 'print_back', 'mark_code_front_anal', 'mark_code_back_anal']  # 각인 필드와 마크 설명 필드
imprint_fields = {c: tuple(normalize(im.get(key)) for key in IMPRINT_KEYS) for c, im in aihub_pill_info.items()}

files_by_imprint = collections.defaultdict(set)  # 기록표의 각인 → test 파일 번호
look_by_imprint = {}  # 기록표의 각인 → test에서 본 색과 모양
imprint_matches = {}  # 기록표의 각인 → AI Hub에서 찾은 K-코드
for row in review_rows:
    if row['imprint'] not in ('', '?'):  # 각인을 읽지 못한 알약은 글자로 대조할 수 없어 뺀다
        files_by_imprint[row['imprint']].add(int(row['test_file'].split('.')[0]))
        look_by_imprint.setdefault(row['imprint'], f"{row['color']} {row['shape']}")

rows = []
for seen, numbers in sorted(files_by_imprint.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    query = normalize(seen.replace('|', '분할선'))
    match, hits = '같음', [c for c, fields in imprint_fields.items() if query in fields]
    if not hits and len(query) >= 2:
        match, hits = '부분 일치', [c for c, fields in imprint_fields.items() if any(query in f for f in fields)]
    imprint_matches[seen] = hits
    shown = ', '.join(map(str, sorted(numbers)[:6])) + (' …' if len(numbers) > 6 else '')
    for c in hits:
        info = aihub_pill_info[c]
        rows.append([seen, look_by_imprint[seen], shown, f'K-{c:06d}', info['dl_name'], f"{info['color_class1']} {info['drug_shape']}",
                     match, '있음' if c in train_classes else '없음', aihub_object_count[c]])
    if not hits:
        rows.append([seen, look_by_imprint[seen], shown, '찾지 못함', '-', '-', '-', '-', '-'])
rows.append([f'**각인 필드가 있는 클래스 {filled}/{len(aihub_pill_info)}**', '', '', '', '', '', '', '', ''])
show_table(['test에서 읽은 각인', 'test 색, 모양', 'test 파일 번호', 'AI Hub K-코드', '제품명', 'AI Hub 색, 모양', '일치',
            'train에 있음', 'AI Hub 객체 수'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 5-2. test 알약과 AI Hub 알약 나란히 보기
- 각인마다 한 줄에 왼쪽 2칸은 test 사진, 오른쪽 4칸은 5-1에서 찾은 AI Hub 클래스의 crop이다.
    - test 사진은 00 5-3처럼 기록표의 위치 쪽 모서리에서 620×780픽셀을 자른다. 각인마다 서로 다른 조합의 사진 2장을 고른다.
    - AI Hub crop은 그 클래스의 정제 라벨 box를 정사각형으로 자른 것이다. 서로 다른 조합에서 4개를 고른다. (seed 고정)
- 같은 알약인지 색, 모양, 크기, 각인으로 대조한다.

In [ ]:
POS_CROP_W, POS_CROP_H = 620, 780  # 기록표 위치 쪽 모서리에서 자를 크기 (00 5-3과 같음)
N_AIHUB = 4  # 각인마다 보여 줄 AI Hub crop 수


def crop_position(image, position):
    """기록표의 위치(예: 왼쪽 위) 쪽 모서리에서 POS_CROP_W×POS_CROP_H 크기로 자른다."""
    x0 = 0 if '왼쪽' in position else IMG_W - POS_CROP_W
    y0 = 0 if '위' in position else IMG_H - POS_CROP_H
    return image.crop((x0, y0, x0 + POS_CROP_W, y0 + POS_CROP_H))


def test_examples(seen, n=2):
    """기록표에서 그 각인이 나오는 줄을 서로 다른 조합에서 n개 고른다. (번호 3개가 한 조합, 00 5-2)"""
    first_of_combo = {}
    for row in sorted((r for r in review_rows if r['imprint'] == seen), key=lambda r: int(r['test_file'].split('.')[0])):
        first_of_combo.setdefault((int(row['test_file'].split('.')[0]) - 1) // 3, row)
    return list(first_of_combo.values())[:n]


aihub_class_anns = collections.defaultdict(list)  # K-코드 → (이미지, annotation) 목록
for fn in sorted(aihub_clean_records):
    for ann in aihub_clean_records[fn]['annotations']:
        aihub_class_anns[ann['category_id']].append((fn, ann))


def aihub_examples(code, n=N_AIHUB, seed=SEED):
    """그 클래스의 annotation을 서로 다른 조합에서 n개 무작위로 고른다."""
    by_combo = {}
    for fn, ann in aihub_class_anns[code]:
        by_combo.setdefault(combo_name(fn), []).append((fn, ann))
    combos = sorted(by_combo)
    picks = np.random.default_rng(seed).choice(len(combos), size=min(n, len(combos)), replace=False)
    return [by_combo[combos[i]][0] for i in sorted(picks)]


lines_to_show = [(seen, c) for seen in sorted(files_by_imprint, key=lambda s: (-len(files_by_imprint[s]), s))
                 for c in (imprint_matches[seen] or [None])]
fig, axes = plt.subplots(len(lines_to_show), 2 + N_AIHUB, figsize=(20, 3.4 * len(lines_to_show)), layout='constrained',
                         gridspec_kw={'width_ratios': [1.6, 1.6] + [1] * N_AIHUB})
for row_axes, (seen, c) in zip(axes, lines_to_show):
    for ax, row in zip(row_axes[:2], test_examples(seen)):
        ax.imshow(crop_position(Image.open(TEST_IMG_DIR / row['test_file']).convert('RGB'), row['position']))
        ax.set_title(f"test {row['test_file']} {row['position']}\n기록표: {seen} ({row['color']} {row['shape']})", fontsize=9)
    if c is None:
        row_axes[2].text(0.5, 0.5, 'AI Hub 각인 필드에서\n찾지 못함', ha='center', va='center', fontsize=12)
    else:
        for ax, (fn, ann) in zip(row_axes[2:], aihub_examples(c)):
            ax.imshow(crop_square(Image.open(AIHUB_IMG_DIR / fn).convert('RGB'), ann['bbox']))
            ax.set_title(f"AI Hub {pill_label(ann)}\n{ann['image']['color_class1']} {ann['image']['drug_shape']}", fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('train에 없는 test 알약(왼쪽 2칸, 기록표 위치 쪽 모서리)과 각인으로 찾은 AI Hub 알약(오른쪽 4칸, 정제 라벨 box)', fontsize=14)
save_fig(fig, '5-2_test_vs_aihub')
plt.show()

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

### 5-3. test 사진 단위 커버리지
- test 사진마다 기록표에 적힌 train에 없는 알약이 모두 학습 클래스로 채워지는지 센다.
    - 기록표에 줄이 없는 test 사진은 모든 알약이 train 갤러리에 있는 사진이다. (00 5-2)
    - 보충 후에는 5-1에서 K-코드를 찾고 5-2에서 같은 알약으로 확인한 각인을 채워진 것으로 본다.
- 목적: test 점수에서 보충으로도 맞힐 수 없는 알약이 얼마나 남는지 확인한다.

In [ ]:
code_of_imprint = {seen: hits[0] for seen, hits in imprint_matches.items() if len(hits) == 1}  # 5-2에서 같은 알약으로 확인한 각인
new_pills_by_file = collections.defaultdict(set)  # test 파일 → 그 사진의 train에 없는 알약 각인
for row in review_rows:
    new_pills_by_file[row['test_file']].add(row['imprint'])
covered_after = [f for f, imprints in new_pills_by_file.items() if all(i in code_of_imprint for i in imprints)]
left = collections.Counter(i for f, imprints in new_pills_by_file.items() for i in imprints if i not in code_of_imprint)
n_test = len(test_files)
rows = [
    ['모든 알약이 학습 클래스에 있는 test 사진', f'{n_test - len(new_pills_by_file)}장 ({(n_test - len(new_pills_by_file)) / n_test:.1%})',
     f'{n_test - len(new_pills_by_file) + len(covered_after)}장 ({(n_test - len(new_pills_by_file) + len(covered_after)) / n_test:.1%})'],
    ['train에 없는 알약이 하나라도 있는 test 사진', f'{len(new_pills_by_file)}장', f'{len(new_pills_by_file) - len(covered_after)}장'],
    ['학습 클래스에 없는 알약 (각인: 나오는 test 사진 수)', f'{len(files_by_imprint)}종',
     ', '.join(f'{i}: {len({r["test_file"] for r in review_rows if r["imprint"] == i})}장' for i in sorted(left)) or '없음'],
]
show_table(['항목', '보충 전 (대회 train만)', '보충 후 (+ AI Hub)'], rows)

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

## 6. 평가 기준선
- 목적: 대회 train과 AI Hub를 합친 118개 클래스 체계로도 로컬 평가 코드가 맞게 동작하는지 확인한다.

### 6-1. 변환 라벨로 mAP 계산 검증
- 00 8-1의 `coco_map`(pycocotools, IoU 기준 0.75~0.95)을 그대로 쓴다.
- 정답은 정제한 대회 train(00 2-5)과 정제한 AI Hub(2-5)를 합친 라벨이다.
- 정답 box를 그대로 예측으로 넣으면 1.0, 클래스를 모두 틀리게 넣으면 0이 나와야 한다.
    - AI Hub 라벨의 category_id를 drug_N으로 만들었으므로(1-3), 대회 train과 같은 알약이 같은 번호로 묶여야 1.0이 된다.

In [ ]:
def coco_map(gt_objects, pred_objects):
    """mAP@[0.75:0.95]를 계산한다. 정답은 (파일명, K-코드, xywh), 예측은 (파일명, K-코드, xywh, 점수)이다. (00 8-1과 같음)"""
    image_ids = {fn: i + 1 for i, fn in enumerate(sorted({o[0] for o in gt_objects}))}
    categories = sorted({o[1] for o in gt_objects} | {o[1] for o in pred_objects})
    gt = {
        'images': [{'id': i, 'width': IMG_W, 'height': IMG_H} for i in image_ids.values()],
        'annotations': [{'id': k + 1, 'image_id': image_ids[fn], 'category_id': c, 'bbox': list(b),
                         'area': b[2] * b[3], 'iscrowd': 0} for k, (fn, c, b) in enumerate(gt_objects)],
        'categories': [{'id': c} for c in categories],
    }
    if not pred_objects:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):  # pycocotools의 진행 메시지 숨김
        coco_gt = COCO()
        coco_gt.dataset = gt
        coco_gt.createIndex()
        coco_dt = coco_gt.loadRes([{'image_id': image_ids[fn], 'category_id': c, 'bbox': list(b), 'score': s}
                                   for fn, c, b, s in pred_objects])
        evaluator = COCOeval(coco_gt, coco_dt, 'bbox')
        evaluator.params.iouThrs = np.linspace(0.75, 0.95, 5)  # 대회 지표의 IoU 기준
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return float(evaluator.stats[0])  # stats[0]: 모든 IoU 기준과 클래스의 AP 평균


eval_objects = ([(fn, a['category_id'], a['bbox']) for fn, r in sorted(clean_records.items()) for a in r['annotations']]
                + [(fn, a['category_id'], a['bbox']) for fn, r in sorted(aihub_clean_records.items()) for a in r['annotations']])
eval_classes = sorted({c for _, c, _ in eval_objects})
wrong_class = {c: eval_classes[(i + 1) % len(eval_classes)] for i, c in enumerate(eval_classes)}
show_table(['예측', 'mAP@[0.75:0.95]'], [
    [f'정답 box와 클래스를 그대로 넣음 (이미지 {len({o[0] for o in eval_objects}):,}장, 객체 {len(eval_objects):,}개, 클래스 {len(eval_classes)}개)',
     f'{coco_map(eval_objects, [(*o, 1.0) for o in eval_objects]):.3f}'],
    ['box는 그대로, 클래스를 모두 다른 클래스로 바꿈', f'{coco_map(eval_objects, [(fn, wrong_class[c], b, 1.0) for fn, c, b in eval_objects]):.3f}'],
    ['예측 없음', f'{coco_map(eval_objects, []):.3f}'],
])

**인사이트**
- (실행 후 작성)

**결론**
- (실행 후 작성)

## 7. EDA 결론
- 1~6번의 결과를 항목별 결정 사항으로 정리한다. 괄호 안 번호는 근거가 된 셀의 번호이다.
- 대회 원본 데이터의 결정(입력 크기, 증강, 후처리, 평가 방법 등)은 00 노트북 9절에 있다. 여기에는 AI Hub 보충에 관한 결정만 적는다.

(실행 후 작성)